<a href="https://colab.research.google.com/github/jasunny28/BanglaScript_compiler/blob/main/BanglaScript_compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1 — IMPORTS + TOKEN KINDS
# Bangla Python-Style Compiler
# ============================================================

from collections import deque
import re


# ============================================================
# DATA TYPES
# ============================================================

TYPE_INT = "TYPE_INT"           # পূর্ণসংখ্যা
TYPE_FLOAT = "TYPE_FLOAT"       # দশমিক
TYPE_STRING = "TYPE_STRING"     # স্ট্রিং


# ============================================================
# CONTROL KEYWORDS
# ============================================================

IF = "IF"                       # যদি
ELSE = "ELSE"                   # নাহলে
WHILE = "WHILE"                 # যতক্ষণ


# ============================================================
# FUNCTION KEYWORDS
# ============================================================

FUNCTION = "FUNCTION"           # ফাংশন
RETURN = "RETURN"               # ফেরত


# ============================================================
# OUTPUT
# ============================================================

PRINT = "PRINT"                 # লেখো


# ============================================================
# DATA STRUCTURES
# ============================================================

STACK = "STACK"                 # স্ট্যাক
QUEUE = "QUEUE"                 # কিউ


# ============================================================
# CLASS / OBJECT
# ============================================================

CLASS = "CLASS"                 # ক্লাস
NEW = "NEW"                     # নতুন


# ============================================================
# IDENTIFIERS AND VALUES
# ============================================================

IDENTIFIER = "IDENTIFIER"

INTEGER = "INTEGER"
FLOAT = "FLOAT"
STRING = "STRING"


# ============================================================
# ARITHMETIC OPERATORS
# ============================================================

PLUS = "PLUS"                   # +
MINUS = "MINUS"                 # -
STAR = "STAR"                   # *
SLASH = "SLASH"                 # /


# ============================================================
# ASSIGNMENT AND COMPARISON
# ============================================================

ASSIGN = "ASSIGN"               # =

EQ = "EQ"                       # ==
NE = "NE"                       # !=

GT = "GT"                       # >
LT = "LT"                       # <

GE = "GE"                       # >=
LE = "LE"                       # <=


# ============================================================
# SYMBOLS
# ============================================================

LPAREN = "LPAREN"               # (
RPAREN = "RPAREN"               # )

COMMA = "COMMA"                 # ,
DOT = "DOT"                     # .

COLON = "COLON"                 # :


# ============================================================
# PYTHON-STYLE STRUCTURE TOKENS
# ============================================================

NEWLINE = "NEWLINE"

INDENT = "INDENT"

DEDENT = "DEDENT"


# ============================================================
# END OF FILE
# ============================================================

EOF = "EOF"


print("Cell 1 Loaded Successfully!")

Cell 1 Loaded Successfully!


In [2]:
# ============================================================
# CELL 2 — TOKEN CLASS
# Bangla Python-Style Compiler
#
# A Token is a small bundle of information
# produced by the Lexer.
#
# Python-style Bangla source example:
#
#     পূর্ণসংখ্যা x = 10
#     যদি x > 5:
#         লেখো(x)
#
# Becomes tokens such as:
#
#     TYPE_INT    → "পূর্ণসংখ্যা"
#     IDENTIFIER  → "x"
#     ASSIGN      → "="
#     INTEGER     → "10"
#     NEWLINE
#
# And for blocks:
#
#     IF         → "যদি"
#     IDENTIFIER → "x"
#     GT         → ">"
#     INTEGER    → "5"
#     COLON      → ":"
#     NEWLINE
#     INDENT
#     PRINT      → "লেখো"
#     ...
#     DEDENT
#
# ============================================================


# ============================================================
# TOKEN CLASS
# ============================================================

class Token:

    def __init__(self, kind, lexeme, line, column):

        # ----------------------------------------------------
        # TOKEN TYPE
        #
        # Examples:
        # TYPE_INT
        # IDENTIFIER
        # INTEGER
        # NEWLINE
        # INDENT
        # DEDENT
        # ----------------------------------------------------
        self.kind = kind


        # ----------------------------------------------------
        # ORIGINAL TEXT
        #
        # Examples:
        # "পূর্ণসংখ্যা"
        # "x"
        # "10"
        # "যদি"
        # ":"
        # ----------------------------------------------------
        self.lexeme = lexeme


        # ----------------------------------------------------
        # LINE NUMBER
        #
        # Used for error messages.
        # ----------------------------------------------------
        self.line = line


        # ----------------------------------------------------
        # COLUMN NUMBER
        #
        # Used for error messages.
        # ----------------------------------------------------
        self.column = column


    # ========================================================
    # STRING REPRESENTATION
    #
    # Controls how a token looks when printed.
    # ========================================================

    def __repr__(self):

        return (
            f"Token("
            f"{self.kind}, "
            f"{repr(self.lexeme)}, "
            f"line={self.line}, "
            f"column={self.column}"
            f")"
        )


# ============================================================
# CELL 2 COMPLETE
# ============================================================

print("Token Class Loaded Successfully!")

Token Class Loaded Successfully!


In [3]:
# ============================================================
# CELL 3 — COMPLETE PYTHON-STYLE LEXER
# Bangla Python-Style Compiler
# ============================================================

import unicodedata


# ============================================================
# BANGLA KEYWORDS
# ============================================================

KEYWORDS = {

    # DATA TYPES
    "পূর্ণসংখ্যা": TYPE_INT,
    "দশমিক": TYPE_FLOAT,
    "স্ট্রিং": TYPE_STRING,


    # CONTROL FLOW
    "যদি": IF,
    "নাহলে": ELSE,
    "যতক্ষণ": WHILE,


    # FUNCTION
    "ফাংশন": FUNCTION,
    "ফেরত": RETURN,


    # OUTPUT
    "লেখো": PRINT,


    # DATA STRUCTURES
    "স্ট্যাক": STACK,
    "কিউ": QUEUE,


    # CLASS / OBJECT
    "ক্লাস": CLASS,
    "নতুন": NEW
}


# ============================================================
# LEXER
# ============================================================

class Lexer:


    # ========================================================
    # INITIALIZATION
    # ========================================================

    def __init__(self, source):

        self.source = (
            source
            .replace("\r\n", "\n")
            .replace("\r", "\n")
        )

        self.tokens = []

        self.errors = []


    # ========================================================
    # ADD TOKEN
    # ========================================================

    def add_token(
        self,
        kind,
        lexeme,
        line,
        column
    ):

        self.tokens.append(

            Token(
                kind,
                lexeme,
                line,
                column
            )

        )


    # ========================================================
    # ERROR
    # ========================================================

    def error(
        self,
        line,
        column,
        message
    ):

        self.errors.append(

            f"Lexer Error at line {line}, "
            f"column {column}: {message}"

        )


    # ========================================================
    # IDENTIFIER START
    #
    # Supports:
    #
    # English
    # Bangla
    # Unicode letters
    # _
    # ========================================================

    def is_ident_start(self, ch):

        return (

            ch == "_"

            or

            ch.isalpha()

            or

            unicodedata.category(
                ch
            ).startswith("L")

        )


    # ========================================================
    # IDENTIFIER PART
    #
    # Bangla characters may contain Unicode marks.
    # ========================================================

    def is_ident_part(self, ch):

        category = unicodedata.category(ch)

        return (

            self.is_ident_start(ch)

            or

            ch.isdigit()

            or

            category.startswith("M")

        )


    # ========================================================
    # TOKENIZE ONE LINE
    # ========================================================

    def tokenize_content(

        self,
        text,
        line,
        base_column

    ):


        i = 0

        length = len(text)


        while i < length:


            ch = text[i]

            column = base_column + i


            # ------------------------------------------------
            # WHITESPACE
            # ------------------------------------------------

            if ch in " \t":

                i += 1

                continue


            # ------------------------------------------------
            # COMMENT
            #
            # Python-style:
            #
            # # comment
            # ------------------------------------------------

            if ch == "#":

                break


            # ------------------------------------------------
            # NUMBER
            # ------------------------------------------------

            if ch.isdigit():


                start = i


                while (

                    i < length

                    and

                    text[i].isdigit()

                ):

                    i += 1


                kind = INTEGER


                # FLOAT

                if (

                    i < length

                    and

                    text[i] == "."

                    and

                    i + 1 < length

                    and

                    text[i + 1].isdigit()

                ):


                    kind = FLOAT

                    i += 1


                    while (

                        i < length

                        and

                        text[i].isdigit()

                    ):

                        i += 1


                self.add_token(

                    kind,

                    text[start:i],

                    line,

                    base_column + start

                )


                continue


            # ------------------------------------------------
            # STRING
            # ------------------------------------------------

            if ch in ('"', "'"):


                quote = ch

                start = i

                i += 1


                value = ""

                closed = False


                while i < length:


                    # ESCAPE

                    if (

                        text[i] == "\\"

                        and

                        i + 1 < length

                    ):


                        escaped = text[i + 1]


                        value += {

                            "n": "\n",

                            "t": "\t",

                            "r": "\r"

                        }.get(

                            escaped,

                            escaped

                        )


                        i += 2

                        continue


                    # STRING END

                    if text[i] == quote:


                        i += 1

                        closed = True

                        break


                    value += text[i]

                    i += 1


                if not closed:


                    self.error(

                        line,

                        column,

                        "Unterminated string."

                    )


                else:


                    self.add_token(

                        STRING,

                        value,

                        line,

                        base_column + start

                    )


                continue


            # ------------------------------------------------
            # IDENTIFIER / KEYWORD
            # ------------------------------------------------

            if self.is_ident_start(ch):


                start = i

                i += 1


                while (

                    i < length

                    and

                    self.is_ident_part(text[i])

                ):

                    i += 1


                word = text[start:i]


                kind = KEYWORDS.get(

                    word,

                    IDENTIFIER

                )


                self.add_token(

                    kind,

                    word,

                    line,

                    base_column + start

                )


                continue


            # ------------------------------------------------
            # TWO CHARACTER OPERATORS
            # ------------------------------------------------

            two = text[i:i + 2]


            two_char_tokens = {


                ">=": GE,

                "<=": LE,

                "==": EQ,

                "!=": NE

            }


            if two in two_char_tokens:


                self.add_token(

                    two_char_tokens[two],

                    two,

                    line,

                    column

                )


                i += 2

                continue


            # ------------------------------------------------
            # ONE CHARACTER TOKENS
            # ------------------------------------------------

            one_char_tokens = {


                "+": PLUS,

                "-": MINUS,

                "*": STAR,

                "/": SLASH,

                "=": ASSIGN,

                ">": GT,

                "<": LT,

                "(": LPAREN,

                ")": RPAREN,

                ",": COMMA,

                ".": DOT,

                ":": COLON

            }


            if ch in one_char_tokens:


                self.add_token(

                    one_char_tokens[ch],

                    ch,

                    line,

                    column

                )


                i += 1

                continue


            # ------------------------------------------------
            # INVALID CHARACTER
            # ------------------------------------------------

            self.error(

                line,

                column,

                f"Invalid character {repr(ch)}."

            )


            i += 1


    # ========================================================
    # TOKENIZE COMPLETE SOURCE
    #
    # Handles:
    #
    # NEWLINE
    # INDENT
    # DEDENT
    # ========================================================

    def tokenize(self):


        self.tokens = []

        self.errors = []


        # Python indentation stack

        indent_stack = [0]


        lines = self.source.split("\n")


        for line_number, raw_line in enumerate(

            lines,

            1

        ):


            # ------------------------------------------------
            # Convert TAB to 4 spaces
            # ------------------------------------------------

            expanded_line = raw_line.replace(

                "\t",

                "    "

            )


            stripped_line = expanded_line.lstrip(" ")


            # ------------------------------------------------
            # EMPTY LINE
            # ------------------------------------------------

            if stripped_line == "":

                continue


            # ------------------------------------------------
            # COMMENT ONLY LINE
            # ------------------------------------------------

            if stripped_line.startswith("#"):

                continue


            # ------------------------------------------------
            # CALCULATE INDENT
            # ------------------------------------------------

            indent = (

                len(expanded_line)

                -

                len(stripped_line)

            )


            # ------------------------------------------------
            # INDENT
            # ------------------------------------------------

            if indent > indent_stack[-1]:


                indent_stack.append(indent)


                self.add_token(

                    INDENT,

                    "",

                    line_number,

                    1

                )


            # ------------------------------------------------
            # DEDENT
            # ------------------------------------------------

            else:


                while (

                    indent < indent_stack[-1]

                ):


                    indent_stack.pop()


                    self.add_token(

                        DEDENT,

                        "",

                        line_number,

                        1

                    )


                # Invalid indentation

                if indent != indent_stack[-1]:


                    self.error(

                        line_number,

                        1,

                        "Inconsistent indentation."

                    )


            # ------------------------------------------------
            # TOKENIZE CONTENT
            # ------------------------------------------------

            self.tokenize_content(

                stripped_line,

                line_number,

                indent + 1

            )


            # ------------------------------------------------
            # END OF LINE
            # ------------------------------------------------

            self.add_token(

                NEWLINE,

                "",

                line_number,

                len(expanded_line) + 1

            )


        # ----------------------------------------------------
        # FINAL DEDENTS
        # ----------------------------------------------------

        while len(indent_stack) > 1:


            indent_stack.pop()


            self.add_token(

                DEDENT,

                "",

                len(lines) + 1,

                1

            )


        # ----------------------------------------------------
        # EOF
        # ----------------------------------------------------

        self.add_token(

            EOF,

            "",

            len(lines) + 1,

            1

        )


        return self.tokens


print("Python-Style Lexer Loaded Successfully!")

Python-Style Lexer Loaded Successfully!


In [4]:
# ============================================================
# CELL 4 — PYTHON-STYLE LEXER TEST
# Bangla Python-Style Compiler
# ============================================================


TEST_SOURCE = '''

# ভেরিয়েবল ঘোষণা

পূর্ণসংখ্যা x = 10
দশমিক y = 2.5
স্ট্রিং নাম = "বাংলা কম্পাইলার"


# গাণিতিক অপারেশন

x = x + 5 * 2


# যদি - নাহলে

যদি x >= 10:
    লেখো(নাম)
নাহলে:
    লেখো("মান ছোট")


# যতক্ষণ

যতক্ষণ x < 13:
    লেখো(x)
    x = x + 1

'''


# ============================================================
# LEXER তৈরি
# ============================================================

lexer = Lexer(TEST_SOURCE)


# ============================================================
# TOKEN তৈরি
# ============================================================

tokens = lexer.tokenize()


# ============================================================
# TOKEN দেখানো
# ============================================================

print("=" * 60)
print("টোকেনসমূহ")
print("=" * 60)


for token in tokens:
    print(token)


# ============================================================
# LEXER ERROR
# ============================================================

print("\n" + "=" * 60)
print("লেক্সার ত্রুটি")
print("=" * 60)


if lexer.errors:

    for error in lexer.errors:
        print(error)

else:

    print("কোনো লেক্সার ত্রুটি পাওয়া যায়নি! ✅")


# ============================================================
# TOTAL
# ============================================================

print("\n" + "=" * 60)

print(
    "মোট টোকেন:",
    len(tokens)
)

print("=" * 60)


print(
    "\nCELL 4 TEST FINISHED! 🎉"
)

টোকেনসমূহ
Token(TYPE_INT, 'পূর্ণসংখ্যা', line=5, column=1)
Token(IDENTIFIER, 'x', line=5, column=13)
Token(ASSIGN, '=', line=5, column=15)
Token(INTEGER, '10', line=5, column=17)
Token(NEWLINE, '', line=5, column=19)
Token(TYPE_FLOAT, 'দশমিক', line=6, column=1)
Token(IDENTIFIER, 'y', line=6, column=7)
Token(ASSIGN, '=', line=6, column=9)
Token(FLOAT, '2.5', line=6, column=11)
Token(NEWLINE, '', line=6, column=14)
Token(TYPE_STRING, 'স্ট্রিং', line=7, column=1)
Token(IDENTIFIER, 'নাম', line=7, column=9)
Token(ASSIGN, '=', line=7, column=13)
Token(STRING, 'বাংলা কম্পাইলার', line=7, column=15)
Token(NEWLINE, '', line=7, column=32)
Token(IDENTIFIER, 'x', line=12, column=1)
Token(ASSIGN, '=', line=12, column=3)
Token(IDENTIFIER, 'x', line=12, column=5)
Token(PLUS, '+', line=12, column=7)
Token(INTEGER, '5', line=12, column=9)
Token(STAR, '*', line=12, column=11)
Token(INTEGER, '2', line=12, column=13)
Token(NEWLINE, '', line=12, column=14)
Token(IF, 'যদি', line=17, column=1)
Token(IDENTIFIE

In [5]:
# ============================================================
# CELL 5 — AST NODES (PART 1)
# Bangla Python-Style Compiler
# ============================================================


# ============================================================
# BASE AST NODE
# ============================================================

class ASTNode:
    """
    সকল AST Node-এর Base Class।
    """
    pass


# ============================================================
# PROGRAM NODE
#
# সম্পূর্ণ Bangla program প্রতিনিধিত্ব করে।
# ============================================================

class Program(ASTNode):

    def __init__(self, statements):

        self.statements = statements


    def __repr__(self):

        return f"Program({self.statements})"


# ============================================================
# BLOCK NODE
#
# Python-style indentation block প্রতিনিধিত্ব করে।
#
# উদাহরণ:
#
# যদি x > 5:
#     লেখো("বড়")
#     লেখো(x)
# ============================================================

class Block(ASTNode):

    def __init__(self, statements):

        self.statements = statements


    def __repr__(self):

        return f"Block({self.statements})"


# ============================================================
# EXPRESSION NODES
# ============================================================


# ------------------------------------------------------------
# NUMBER
#
# উদাহরণ:
#
# 10
# 2.5
# ------------------------------------------------------------

class Number(ASTNode):

    def __init__(self, value):

        self.value = value


    def __repr__(self):

        return f"Number({self.value})"


# ------------------------------------------------------------
# STRING
#
# উদাহরণ:
#
# "বাংলা"
# ------------------------------------------------------------

class String(ASTNode):

    def __init__(self, value):

        self.value = value


    def __repr__(self):

        return f"String({repr(self.value)})"


# ------------------------------------------------------------
# VARIABLE
#
# উদাহরণ:
#
# x
# নাম
# ফল
# ------------------------------------------------------------

class Variable(ASTNode):

    def __init__(self, name):

        self.name = name


    def __repr__(self):

        return f"Variable({self.name})"


# ------------------------------------------------------------
# BINARY OPERATION
#
# উদাহরণ:
#
# a + b
# x * y
# x >= 10
# ------------------------------------------------------------

class BinaryOp(ASTNode):

    def __init__(self, left, operator, right):

        self.left = left
        self.operator = operator
        self.right = right


    def __repr__(self):

        return (
            f"BinaryOp("
            f"{self.left}, "
            f"{self.operator}, "
            f"{self.right}"
            f")"
        )


# ------------------------------------------------------------
# UNARY OPERATION
#
# উদাহরণ:
#
# -x
# ------------------------------------------------------------

class UnaryOp(ASTNode):

    def __init__(self, operator, operand):

        self.operator = operator
        self.operand = operand


    def __repr__(self):

        return (
            f"UnaryOp("
            f"{self.operator}, "
            f"{self.operand}"
            f")"
        )


# ------------------------------------------------------------
# FUNCTION CALL
#
# উদাহরণ:
#
# যোগ(10, 20)
# ------------------------------------------------------------

class FunctionCall(ASTNode):

    def __init__(self, name, arguments):

        self.name = name
        self.arguments = arguments


    def __repr__(self):

        return (
            f"FunctionCall("
            f"{self.name}, "
            f"{self.arguments}"
            f")"
        )


# ------------------------------------------------------------
# MEMBER ACCESS
#
# ব্যবহার:
#
# ছাত্র.নাম
# s.ঠেলো(10)
# q.ঢোকাও("A")
# ------------------------------------------------------------

class MemberAccess(ASTNode):

    def __init__(self, object_expr, member):

        self.object_expr = object_expr
        self.member = member


    def __repr__(self):

        return (
            f"MemberAccess("
            f"{self.object_expr}, "
            f"{self.member}"
            f")"
        )


# ------------------------------------------------------------
# NEW OBJECT
#
# উদাহরণ:
#
# নতুন শিক্ষার্থী()
# ------------------------------------------------------------

class NewObject(ASTNode):

    def __init__(self, class_name):

        self.class_name = class_name


    def __repr__(self):

        return f"NewObject({self.class_name})"

In [6]:
# ============================================================
# CELL 6 — AST NODES (PART 2)
# STATEMENT NODES
# Bangla Python-Style Compiler
# ============================================================


# ============================================================
# VARIABLE DECLARATION
#
# উদাহরণ:
#
# পূর্ণসংখ্যা x = 10
# দশমিক y = 2.5
# স্ট্রিং নাম = "Bangla"
# ============================================================

class VarDecl(ASTNode):

    def __init__(self, var_type, name, value):

        self.var_type = var_type
        self.name = name
        self.value = value


    def __repr__(self):

        return (
            f"VarDecl("
            f"{self.var_type}, "
            f"{self.name}, "
            f"{self.value}"
            f")"
        )


# ============================================================
# ASSIGNMENT
#
# উদাহরণ:
#
# x = x + 5
# ছাত্র.নাম = "রহিম"
# ============================================================

class Assignment(ASTNode):

    def __init__(self, target, value):

        self.target = target
        self.value = value


    def __repr__(self):

        return (
            f"Assignment("
            f"{self.target}, "
            f"{self.value}"
            f")"
        )


# ============================================================
# PRINT STATEMENT
#
# উদাহরণ:
#
# লেখো(x)
# লেখো("Hello")
# ============================================================

class PrintStatement(ASTNode):

    def __init__(self, expression):

        self.expression = expression


    def __repr__(self):

        return f"PrintStatement({self.expression})"


# ============================================================
# IF STATEMENT
#
# Python-style Bangla syntax:
#
# যদি x > 5:
#     লেখো("বড়")
# নাহলে:
#     লেখো("ছোট")
# ============================================================

class IfStatement(ASTNode):

    def __init__(
        self,
        condition,
        then_branch,
        else_branch=None
    ):

        self.condition = condition
        self.then_branch = then_branch
        self.else_branch = else_branch


    def __repr__(self):

        return (
            f"IfStatement("
            f"{self.condition}, "
            f"{self.then_branch}, "
            f"{self.else_branch}"
            f")"
        )


# ============================================================
# WHILE STATEMENT
#
# Python-style Bangla syntax:
#
# যতক্ষণ x < 10:
#     x = x + 1
# ============================================================

class WhileStatement(ASTNode):

    def __init__(self, condition, body):

        self.condition = condition
        self.body = body


    def __repr__(self):

        return (
            f"WhileStatement("
            f"{self.condition}, "
            f"{self.body}"
            f")"
        )


# ============================================================
# FUNCTION DECLARATION
#
# Python-style Bangla syntax:
#
# ফাংশন যোগ(পূর্ণসংখ্যা a, পূর্ণসংখ্যা b):
#     ফেরত a + b
#
# parameters =
#
# [
#     ("TYPE_INT", "a"),
#     ("TYPE_INT", "b")
# ]
# ============================================================

class FunctionDecl(ASTNode):

    def __init__(
        self,
        name,
        parameters,
        body
    ):

        self.name = name
        self.parameters = parameters
        self.body = body


    def __repr__(self):

        return (
            f"FunctionDecl("
            f"{self.name}, "
            f"{self.parameters}, "
            f"{self.body}"
            f")"
        )


# ============================================================
# RETURN STATEMENT
#
# উদাহরণ:
#
# ফেরত a + b
# ============================================================

class ReturnStatement(ASTNode):

    def __init__(self, value):

        self.value = value


    def __repr__(self):

        return f"ReturnStatement({self.value})"


# ============================================================
# STACK DECLARATION
#
# উদাহরণ:
#
# স্ট্যাক s
# ============================================================

class StackDecl(ASTNode):

    def __init__(self, name):

        self.name = name


    def __repr__(self):

        return f"StackDecl({self.name})"


# ============================================================
# QUEUE DECLARATION
#
# উদাহরণ:
#
# কিউ q
# ============================================================

class QueueDecl(ASTNode):

    def __init__(self, name):

        self.name = name


    def __repr__(self):

        return f"QueueDecl({self.name})"


# ============================================================
# CLASS DECLARATION
#
# Simplified Python-style Bangla Class:
#
# ক্লাস শিক্ষার্থী:
#     স্ট্রিং নাম = "অজানা"
#     পূর্ণসংখ্যা বয়স = 0
#
# শুধুমাত্র properties থাকবে।
#
# Methods, Constructor, Inheritance এবং Polymorphism
# এই locked plan-এর অংশ নয়।
# ============================================================

class ClassDecl(ASTNode):

    def __init__(self, name, properties):

        self.name = name

        # properties-এর মধ্যে VarDecl node থাকবে
        self.properties = properties


    def __repr__(self):

        return (
            f"ClassDecl("
            f"{self.name}, "
            f"{self.properties}"
            f")"
        )


# ============================================================
# EXPRESSION STATEMENT
#
# যখন কোনো expression statement হিসেবে ব্যবহৃত হয়।
#
# উদাহরণ:
#
# যোগ(10, 20)
# s.ঠেলো(100)
# q.ঢোকাও("A")
# ============================================================

class ExpressionStatement(ASTNode):

    def __init__(self, expression):

        self.expression = expression


    def __repr__(self):

        return (
            f"ExpressionStatement("
            f"{self.expression}"
            f")"
        )

In [7]:
# ============================================================
# CELL 7 — AST BASIC TEST
# Bangla Python-Style Compiler
# ============================================================
#
# এই Cell-এ AST Node গুলো Python-style Bangla syntax অনুযায়ী
# পরীক্ষা করা হচ্ছে।
#
# উদাহরণ:
#
# পূর্ণসংখ্যা x = 10
#
# যদি x > 5:
#     লেখো("বড়")
#
# স্ট্যাক s
#
# ============================================================


print("========== AST TEST START ==========\n")


# ============================================================
# ১. NUMBER
# ============================================================

num = Number(10)

print("১. Number:")
print(num)


# ============================================================
# ২. STRING
# ============================================================

text = String("বাংলা")

print("\n২. String:")
print(text)


# ============================================================
# ৩. ARITHMETIC EXPRESSION
#
# Bangla Source:
#
# 10 + 5 * 2
# ============================================================

expr = BinaryOp(
    Number(10),
    "+",
    BinaryOp(
        Number(5),
        "*",
        Number(2)
    )
)

print("\n৩. Arithmetic Expression:")
print(expr)


# ============================================================
# ৪. VARIABLE DECLARATION
#
# Bangla Python-style Source:
#
# পূর্ণসংখ্যা x = 10
# ============================================================

var_decl = VarDecl(
    TYPE_INT,
    "x",
    Number(10)
)

print("\n৪. Variable Declaration:")
print(var_decl)


# ============================================================
# ৫. ASSIGNMENT
#
# Bangla Python-style Source:
#
# x = x + 5
# ============================================================

assignment = Assignment(
    "x",
    BinaryOp(
        Variable("x"),
        "+",
        Number(5)
    )
)

print("\n৫. Assignment:")
print(assignment)


# ============================================================
# ৬. PRINT STATEMENT
#
# Bangla Source:
#
# লেখো("বাংলা")
# ============================================================

print_node = PrintStatement(
    String("বাংলা")
)

print("\n৬. Print Statement:")
print(print_node)


# ============================================================
# ৭. IF STATEMENT
#
# Bangla Python-style Source:
#
# যদি x > 5:
#     লেখো("বড়")
# ============================================================

if_node = IfStatement(

    BinaryOp(
        Variable("x"),
        ">",
        Number(5)
    ),

    Block([
        PrintStatement(
            String("বড়")
        )
    ])
)

print("\n৭. If Statement:")
print(if_node)


# ============================================================
# ৮. IF ELSE STATEMENT
#
# Bangla Python-style Source:
#
# যদি x > 5:
#     লেখো("বড়")
# নাহলে:
#     লেখো("ছোট")
# ============================================================

if_else_node = IfStatement(

    BinaryOp(
        Variable("x"),
        ">",
        Number(5)
    ),

    Block([
        PrintStatement(
            String("বড়")
        )
    ]),

    Block([
        PrintStatement(
            String("ছোট")
        )
    ])
)

print("\n৮. If Else Statement:")
print(if_else_node)


# ============================================================
# ৯. WHILE STATEMENT
#
# Bangla Python-style Source:
#
# যতক্ষণ x < 10:
#     x = x + 1
# ============================================================

while_node = WhileStatement(

    BinaryOp(
        Variable("x"),
        "<",
        Number(10)
    ),

    Block([
        Assignment(
            "x",
            BinaryOp(
                Variable("x"),
                "+",
                Number(1)
            )
        )
    ])
)

print("\n৯. While Statement:")
print(while_node)


# ============================================================
# ১০. STACK DECLARATION
#
# Bangla Python-style Source:
#
# স্ট্যাক s
# ============================================================

stack_node = StackDecl(
    "s"
)

print("\n১০. Stack Declaration:")
print(stack_node)


# ============================================================
# ১১. QUEUE DECLARATION
#
# Bangla Python-style Source:
#
# কিউ q
# ============================================================

queue_node = QueueDecl(
    "q"
)

print("\n১১. Queue Declaration:")
print(queue_node)


# ============================================================
# FINAL RESULT
# ============================================================

print("\n========== AST TEST FINISHED ==========")
print("CELL 7 সফলভাবে Python-style AST test সম্পন্ন করেছে! 🎉")

========== AST TEST START ==========

১. Number:
Number(10)

২. String:
String('বাংলা')

৩. Arithmetic Expression:
BinaryOp(Number(10), +, BinaryOp(Number(5), *, Number(2)))

৪. Variable Declaration:
VarDecl(TYPE_INT, x, Number(10))

৫. Assignment:
Assignment(x, BinaryOp(Variable(x), +, Number(5)))

৬. Print Statement:
PrintStatement(String('বাংলা'))

৭. If Statement:
IfStatement(BinaryOp(Variable(x), >, Number(5)), Block([PrintStatement(String('বড়'))]), None)

৮. If Else Statement:
IfStatement(BinaryOp(Variable(x), >, Number(5)), Block([PrintStatement(String('বড়'))]), Block([PrintStatement(String('ছোট'))]))

৯. While Statement:
WhileStatement(BinaryOp(Variable(x), <, Number(10)), Block([Assignment(x, BinaryOp(Variable(x), +, Number(1)))]))

১০. Stack Declaration:
StackDecl(s)

১১. Queue Declaration:
QueueDecl(q)

========== AST TEST FINISHED ==========
CELL 7 সফলভাবে Python-style AST test সম্পন্ন করেছে! 🎉


In [8]:
# ============================================================
# CELL 8 — PARSER BASE + HELPER METHODS
# Bangla Python-Style Compiler
# ============================================================


# ============================================================
# PARSER ERROR
# ============================================================

class ParserError(Exception):

    pass


# ============================================================
# PARSER
# ============================================================

class Parser:


    def __init__(self, tokens):


        self.tokens = tokens


        self.current = 0


        self.errors = []


    # ========================================================
    # CURRENT TOKEN
    # ========================================================

    def peek(self):

        return self.tokens[self.current]


    # ========================================================
    # PREVIOUS TOKEN
    # ========================================================

    def previous(self):

        return self.tokens[

            max(

                0,

                self.current - 1

            )

        ]


    # ========================================================
    # EOF CHECK
    # ========================================================

    def is_at_end(self):

        return (

            self.peek().kind == EOF

        )


    # ========================================================
    # ADVANCE
    # ========================================================

    def advance(self):


        if not self.is_at_end():

            self.current += 1


        return self.previous()


    # ========================================================
    # CHECK TOKEN
    # ========================================================

    def check(self, kind):


        if self.is_at_end():

            return False


        return (

            self.peek().kind == kind

        )


    # ========================================================
    # MATCH TOKEN
    # ========================================================

    def match(self, *kinds):


        for kind in kinds:


            if self.check(kind):


                self.advance()


                return True


        return False


    # ========================================================
    # CONSUME REQUIRED TOKEN
    # ========================================================

    def consume(

        self,

        kind,

        message

    ):


        if self.check(kind):

            return self.advance()


        raise self.error(

            self.peek(),

            message

        )


    # ========================================================
    # PARSER ERROR
    # ========================================================

    def error(

        self,

        token,

        message

    ):


        error_text = (

            f"Parser Error at line "

            f"{token.line}, "

            f"column {token.column}: "

            f"{message}"

        )


        self.errors.append(

            error_text

        )


        return ParserError(

            error_text

        )


    # ========================================================
    # SKIP EMPTY NEWLINES
    # ========================================================

    def skip_newlines(self):


        while self.match(NEWLINE):

            pass


    # ========================================================
    # END OF STATEMENT
    #
    # Python-style statement ends at NEWLINE
    # ========================================================

    def consume_statement_end(self):


        if self.match(NEWLINE):


            self.skip_newlines()


            return


        # End of block or file is also valid

        if (

            self.check(DEDENT)

            or

            self.check(EOF)

        ):

            return


        raise self.error(

            self.peek(),

            "Expected end of statement."

        )


    # ========================================================
    # BLOCK START
    #
    # :
    # NEWLINE
    # INDENT
    # ========================================================

    def consume_suite_start(

        self,

        message

    ):


        self.consume(

            COLON,

            message

        )


        self.consume(

            NEWLINE,

            "Expected new line after ':'."

        )


        self.skip_newlines()


        self.consume(

            INDENT,

            "Expected indented block."

        )


print(

    "Python-Style Parser Base Loaded Successfully!"

)

Python-Style Parser Base Loaded Successfully!


In [9]:
# ============================================================
# CELL 9 — EXPRESSION PARSER + OPERATOR PRECEDENCE
# ============================================================


# ============================================================
# EXPRESSION
# ============================================================

def expression(self):

    return self.equality()


# ============================================================
# EQUALITY
#
# == !=
# ============================================================

def equality(self):


    expr = self.comparison()


    while self.match(

        EQ,

        NE

    ):


        operator = self.previous()


        right = self.comparison()


        expr = BinaryOp(

            expr,

            operator.lexeme,

            right

        )


    return expr


# ============================================================
# COMPARISON
#
# > < >= <=
# ============================================================

def comparison(self):


    expr = self.term()


    while self.match(

        GT,

        LT,

        GE,

        LE

    ):


        operator = self.previous()


        right = self.term()


        expr = BinaryOp(

            expr,

            operator.lexeme,

            right

        )


    return expr


# ============================================================
# ADDITION / SUBTRACTION
# ============================================================

def term(self):


    expr = self.factor()


    while self.match(

        PLUS,

        MINUS

    ):


        operator = self.previous()


        right = self.factor()


        expr = BinaryOp(

            expr,

            operator.lexeme,

            right

        )


    return expr


# ============================================================
# MULTIPLICATION / DIVISION
# ============================================================

def factor(self):


    expr = self.unary()


    while self.match(

        STAR,

        SLASH

    ):


        operator = self.previous()


        right = self.unary()


        expr = BinaryOp(

            expr,

            operator.lexeme,

            right

        )


    return expr


# ============================================================
# UNARY
#
# -x
# ============================================================

def unary(self):


    if self.match(MINUS):


        operator = self.previous()


        return UnaryOp(

            operator.lexeme,

            self.unary()

        )


    return self.primary()


# ============================================================
# PRIMARY
# ============================================================

def primary(self):


    # INTEGER

    if self.match(INTEGER):


        return Number(

            int(

                self.previous().lexeme

            )

        )


    # FLOAT

    if self.match(FLOAT):


        return Number(

            float(

                self.previous().lexeme

            )

        )


    # STRING

    if self.match(STRING):


        return String(

            self.previous().lexeme

        )


    # NEW OBJECT

    if self.match(NEW):


        class_name = self.consume(

            IDENTIFIER,

            "Expected class name after নতুন."

        ).lexeme


        self.consume(

            LPAREN,

            "Expected '(' after class name."

        )


        self.consume(

            RPAREN,

            "Expected ')' after class name."

        )


        return NewObject(

            class_name

        )


    # IDENTIFIER

    if self.match(IDENTIFIER):


        expr = Variable(

            self.previous().lexeme

        )


        while True:


            # MEMBER ACCESS

            if self.match(DOT):


                member = self.consume(

                    IDENTIFIER,

                    "Expected member name after '.'."

                ).lexeme


                expr = MemberAccess(

                    expr,

                    member

                )


            # FUNCTION / METHOD CALL

            elif self.match(LPAREN):


                arguments = []


                if not self.check(RPAREN):


                    while True:


                        arguments.append(

                            self.expression()

                        )


                        if not self.match(COMMA):

                            break


                self.consume(

                    RPAREN,

                    "Expected ')' after arguments."

                )


                if isinstance(

                    expr,

                    Variable

                ):


                    expr = FunctionCall(

                        expr.name,

                        arguments

                    )


                else:


                    expr = FunctionCall(

                        expr,

                        arguments

                    )


            else:

                break


        return expr


    # GROUPED EXPRESSION

    if self.match(LPAREN):


        expr = self.expression()


        self.consume(

            RPAREN,

            "Expected ')' after expression."

        )


        return expr


    raise self.error(

        self.peek(),

        "Expected expression."

    )


# ============================================================
# ATTACH METHODS TO PARSER
# ============================================================

Parser.expression = expression

Parser.equality = equality

Parser.comparison = comparison

Parser.term = term

Parser.factor = factor

Parser.unary = unary

Parser.primary = primary


print(

    "Expression Parser Loaded Successfully!"

)

Expression Parser Loaded Successfully!


In [10]:
# ============================================================
# CELL 10 — BASIC STATEMENT PARSER
# Python-Style Bangla Compiler
# ============================================================


# ============================================================
# LOOK AHEAD
# ============================================================

def check_next(self, kind):


    next_position = (

        self.current + 1

    )


    if next_position >= len(self.tokens):

        return False


    return (

        self.tokens[
            next_position
        ].kind == kind

    )


# ============================================================
# OBJECT DECLARATION START
#
# শিক্ষার্থী ছাত্র = নতুন শিক্ষার্থী()
# ============================================================

def is_object_declaration_start(self):


    return (


        self.check(IDENTIFIER)

        and

        self.check_next(IDENTIFIER)

        and

        self.current + 2 < len(self.tokens)

        and

        self.tokens[
            self.current + 2
        ].kind == ASSIGN


    )


# ============================================================
# VARIABLE DECLARATION
# ============================================================

def var_declaration(self):


    # Type token was already matched

    var_type = self.previous().kind


    # Variable name

    name = self.consume(

        IDENTIFIER,

        "Expected variable name."

    ).lexeme


    value = None


    # Optional initialization

    if self.match(ASSIGN):


        value = self.expression()


    self.consume_statement_end()


    return VarDecl(

        var_type,

        name,

        value

    )


# ============================================================
# ASSIGNMENT OR EXPRESSION
# ============================================================

def assignment_or_expression(self):


    expr = self.expression()


    # ASSIGNMENT

    if self.match(ASSIGN):


        value = self.expression()


        self.consume_statement_end()


        return Assignment(

            expr,

            value

        )


    # EXPRESSION STATEMENT

    self.consume_statement_end()


    return ExpressionStatement(

        expr

    )


# ============================================================
# PRINT
# ============================================================

def print_statement(self):


    self.consume(

        LPAREN,

        "Expected '(' after লেখো."

    )


    value = self.expression()


    self.consume(

        RPAREN,

        "Expected ')' after print value."

    )


    self.consume_statement_end()


    return PrintStatement(

        value

    )


# ============================================================
# STACK DECLARATION
# ============================================================

def stack_declaration(self):


    name = self.consume(

        IDENTIFIER,

        "Expected stack name."

    ).lexeme


    self.consume_statement_end()


    return StackDecl(

        name

    )


# ============================================================
# QUEUE DECLARATION
# ============================================================

def queue_declaration(self):


    name = self.consume(

        IDENTIFIER,

        "Expected queue name."

    ).lexeme


    self.consume_statement_end()


    return QueueDecl(

        name

    )


# ============================================================
# OBJECT DECLARATION
#
# শিক্ষার্থী ছাত্র = নতুন শিক্ষার্থী()
# ============================================================

def object_declaration(self):


    class_name = self.consume(

        IDENTIFIER,

        "Expected class name."

    ).lexeme


    object_name = self.consume(

        IDENTIFIER,

        "Expected object name."

    ).lexeme


    self.consume(

        ASSIGN,

        "Expected '=' in object declaration."

    )


    value = self.expression()


    self.consume_statement_end()


    return VarDecl(

        class_name,

        object_name,

        value

    )


# ============================================================
# BASIC STATEMENT
# ============================================================

def basic_statement(self):


    # DATA TYPES

    if self.match(

        TYPE_INT,

        TYPE_FLOAT,

        TYPE_STRING

    ):


        return self.var_declaration()


    # PRINT

    if self.match(PRINT):


        return self.print_statement()


    # STACK

    if self.match(STACK):


        return self.stack_declaration()


    # QUEUE

    if self.match(QUEUE):


        return self.queue_declaration()


    # OBJECT DECLARATION

    if self.is_object_declaration_start():


        return self.object_declaration()


    # ASSIGNMENT / EXPRESSION

    return self.assignment_or_expression()


# ============================================================
# ATTACH METHODS
# ============================================================

Parser.check_next = check_next

Parser.is_object_declaration_start = is_object_declaration_start

Parser.var_declaration = var_declaration

Parser.assignment_or_expression = assignment_or_expression

Parser.print_statement = print_statement

Parser.stack_declaration = stack_declaration

Parser.queue_declaration = queue_declaration

Parser.object_declaration = object_declaration

Parser.basic_statement = basic_statement


print(

    "Basic Statement Parser Loaded Successfully!"

)

Basic Statement Parser Loaded Successfully!


In [11]:
# ============================================================
# CELL 11 — PYTHON-STYLE BLOCK + IF ELSE + WHILE
# ============================================================


# ============================================================
# BLOCK
#
# Example:
#
# যদি x > 5:
#     লেখো("বড়")
#
# Lexer provides:
#
# NEWLINE
# INDENT
# ...
# DEDENT
# ============================================================

def block(self):


    statements = []


    self.skip_newlines()


    while (

        not self.check(DEDENT)

        and

        not self.is_at_end()

    ):


        try:


            statement = self.statement()


            if statement is not None:


                statements.append(

                    statement

                )


        except ParserError:


            self.synchronize()


    self.consume(

        DEDENT,

        "Expected end of indented block."

    )


    return Block(

        statements

    )


# ============================================================
# IF STATEMENT
#
# যদি x > 5:
#     লেখো("বড়")
#
# নাহলে:
#     লেখো("ছোট")
# ============================================================

def if_statement(self):


    # CONDITION

    condition = self.expression()


    # :
    # NEWLINE
    # INDENT

    self.consume_suite_start(

        "Expected ':' after IF condition."

    )


    # THEN BLOCK

    then_branch = self.block()


    self.skip_newlines()


    else_branch = None


    # ELSE

    if self.match(ELSE):


        self.consume_suite_start(

            "Expected ':' after ELSE."

        )


        else_branch = self.block()


    return IfStatement(

        condition,

        then_branch,

        else_branch

    )


# ============================================================
# WHILE STATEMENT
#
# যতক্ষণ x < 10:
#     x = x + 1
# ============================================================

def while_statement(self):


    condition = self.expression()


    self.consume_suite_start(

        "Expected ':' after WHILE condition."

    )


    body = self.block()


    return WhileStatement(

        condition,

        body

    )


# ============================================================
# GENERAL STATEMENT
# ============================================================

def statement(self):


    self.skip_newlines()


    # End of file / block

    if (

        self.is_at_end()

        or

        self.check(DEDENT)

    ):

        return None


    # IF

    if self.match(IF):

        return self.if_statement()


    # WHILE

    if self.match(WHILE):

        return self.while_statement()


    # FUNCTION

    if self.match(FUNCTION):

        return self.function_declaration()


    # RETURN

    if self.match(RETURN):

        return self.return_statement()


    # CLASS

    if self.match(CLASS):

        return self.class_declaration()


    # BASIC STATEMENT

    return self.basic_statement()


# ============================================================
# ATTACH METHODS
# ============================================================

Parser.block = block

Parser.if_statement = if_statement

Parser.while_statement = while_statement

Parser.statement = statement


print(

    "Python-Style IF-ELSE, WHILE and Block Parser Loaded Successfully!"

)

Python-Style IF-ELSE, WHILE and Block Parser Loaded Successfully!


In [12]:
# ============================================================
# CELL 12 — FUNCTIONS + CLASS
# Python-Style Bangla Compiler
# ============================================================


# ============================================================
# FUNCTION DECLARATION
#
# ফাংশন যোগ(পূর্ণসংখ্যা a, পূর্ণসংখ্যা b):
#     ফেরত a + b
# ============================================================

def function_declaration(self):


    # FUNCTION NAME

    name = self.consume(

        IDENTIFIER,

        "Expected function name."

    ).lexeme


    # (

    self.consume(

        LPAREN,

        "Expected '(' after function name."

    )


    parameters = []


    # ========================================================
    # PARAMETERS
    # ========================================================

    if not self.check(RPAREN):


        while True:


            # PARAMETER TYPE

            if not self.match(

                TYPE_INT,

                TYPE_FLOAT,

                TYPE_STRING

            ):


                raise self.error(

                    self.peek(),

                    "Expected parameter type."

                )


            parameter_type = (

                self.previous().kind

            )


            # PARAMETER NAME

            parameter_name = self.consume(

                IDENTIFIER,

                "Expected parameter name."

            ).lexeme


            parameters.append(

                (

                    parameter_type,

                    parameter_name

                )

            )


            # MORE PARAMETERS?

            if not self.match(COMMA):

                break


    # )

    self.consume(

        RPAREN,

        "Expected ')' after parameters."

    )


    # :
    # NEWLINE
    # INDENT

    self.consume_suite_start(

        "Expected ':' after function declaration."

    )


    # BODY

    body = self.block()


    return FunctionDecl(

        name,

        parameters,

        body

    )


# ============================================================
# RETURN
#
# ফেরত a + b
# ============================================================

def return_statement(self):


    value = None


    if not self.check(NEWLINE):


        value = self.expression()


    self.consume_statement_end()


    return ReturnStatement(

        value

    )


# ============================================================
# CLASS DECLARATION
#
# ক্লাস শিক্ষার্থী:
#     স্ট্রিং নাম = "অজানা"
#     পূর্ণসংখ্যা বয়স = 0
# ============================================================

def class_declaration(self):


    # CLASS NAME

    name = self.consume(

        IDENTIFIER,

        "Expected class name."

    ).lexeme


    # :
    # NEWLINE
    # INDENT

    self.consume_suite_start(

        "Expected ':' after class name."

    )


    properties = []


    self.skip_newlines()


    # ========================================================
    # CLASS PROPERTIES
    # ========================================================

    while (

        not self.check(DEDENT)

        and

        not self.is_at_end()

    ):


        if self.match(

            TYPE_INT,

            TYPE_FLOAT,

            TYPE_STRING

        ):


            properties.append(

                self.var_declaration()

            )


        else:


            raise self.error(

                self.peek(),

                "Only typed properties are allowed inside class."

            )


    # END CLASS BLOCK

    self.consume(

        DEDENT,

        "Expected end of class block."

    )


    return ClassDecl(

        name,

        properties

    )


# ============================================================
# ATTACH METHODS
# ============================================================

Parser.function_declaration = function_declaration

Parser.return_statement = return_statement

Parser.class_declaration = class_declaration


print(

    "Functions and Class Parser Loaded Successfully!"

)

Functions and Class Parser Loaded Successfully!


In [13]:
# ============================================================
# CELL 13 — PARSER ERROR RECOVERY + FULL PARSE
# Python-Style Bangla Compiler
# ============================================================


# ============================================================
# SYNCHRONIZE
#
# Error
#   ↓
# Skip invalid tokens
#   ↓
# Stop at NEWLINE
# or safe statement boundary
# ============================================================

def synchronize(self):


    if self.is_at_end():

        return


    # Always move forward at least once

    self.advance()


    while not self.is_at_end():


        # ----------------------------------------------------
        # NEWLINE / BLOCK END
        # ----------------------------------------------------

        if self.previous().kind in (

            NEWLINE,

            DEDENT

        ):


            self.skip_newlines()


            return


        # ----------------------------------------------------
        # SAFE STATEMENT BOUNDARIES
        # ----------------------------------------------------

        if self.peek().kind in (


            TYPE_INT,

            TYPE_FLOAT,

            TYPE_STRING,


            IF,

            ELSE,

            WHILE,


            FUNCTION,

            RETURN,


            PRINT,


            STACK,

            QUEUE,


            CLASS,


            IDENTIFIER,


            DEDENT

        ):


            return


        self.advance()


# ============================================================
# FULL PARSE
# ============================================================

def parse(self):


    statements = []


    self.skip_newlines()


    while not self.is_at_end():


        # Unexpected DEDENT at top level

        if self.match(DEDENT):


            token = self.previous()


            self.errors.append(

                f"Parser Error at line "

                f"{token.line}, "

                f"column {token.column}: "

                f"Unexpected indentation end."

            )


            self.skip_newlines()


            continue


        try:


            statement = self.statement()


            if statement is not None:


                statements.append(

                    statement

                )


        except ParserError:


            self.synchronize()


        self.skip_newlines()


    return Program(

        statements

    )


# ============================================================
# ATTACH METHODS
# ============================================================

Parser.synchronize = synchronize

Parser.parse = parse


print(

    "Python-Style Parser Error Recovery Loaded Successfully!"

)

Python-Style Parser Error Recovery Loaded Successfully!


In [14]:
# ============================================================
# CELL 14 — সম্পূর্ণ বাংলা পাইথন-স্টাইল কম্পাইলার পরীক্ষা
# FINAL LOCKED PLAN অনুযায়ী DEMO SOURCE
# ============================================================


# ============================================================
# FINAL DEMO SOURCE
#
# Source Language : বাংলা
# Syntax          : Python-style indentation
# Target          : Python backend
#
# গুরুত্বপূর্ণ:
# - { } নেই
# - ; নেই
# - NEWLINE / INDENT / DEDENT ব্যবহার হবে
# - Boolean / logical operator নেই
# - Class-এ method / constructor / inheritance নেই
# ============================================================

DEMO_SOURCE = '''
# ========================================================
# ১. Data Types
# ========================================================

পূর্ণসংখ্যা x = 10
দশমিক y = 2.5
স্ট্রিং নাম = "বাংলাভাষা"


# ========================================================
# ২. Assignment + ৩. Arithmetic / Expression
# ========================================================

x = x + 5 * 2


# ========================================================
# ৫. Print
# ========================================================

লেখো("বাংলা কম্পাইলারে স্বাগতম")
লেখো(নাম)
লেখো(x)


# ========================================================
# ৭. IF – ELSE
# ========================================================

যদি x > 5:
    লেখো("মান বড়")
নাহলে:
    লেখো("মান ছোট")


# ========================================================
# ৮. WHILE
# ========================================================

যতক্ষণ x < 25:
    লেখো(x)
    x = x + 1


# ========================================================
# ৯. Function
# ========================================================

ফাংশন যোগ(পূর্ণসংখ্যা a, পূর্ণসংখ্যা b):
    ফেরত a + b


পূর্ণসংখ্যা ফল = যোগ(10, 20)

লেখো(ফল)


# ========================================================
# ১০. Stack
# ========================================================

স্ট্যাক s

s.ঠেলো(100)
s.ঠেলো(200)

লেখো(s.উপর_দেখো())
লেখো(s.বের_করো())


# ========================================================
# ১১. Queue
# ========================================================

কিউ q

q.ঢোকাও("ক")
q.ঢোকাও("খ")

লেখো(q.সামনে())
লেখো(q.বের_করো())


# ========================================================
# ১২. Simplified Class / Object
# ========================================================

ক্লাস শিক্ষার্থী:
    স্ট্রিং নাম = "অজানা"
    পূর্ণসংখ্যা বয়স = 0


শিক্ষার্থী ছাত্র = নতুন শিক্ষার্থী()

ছাত্র.নাম = "রহিম"

লেখো(ছাত্র.নাম)
'''


# ============================================================
# Downstream Cells Compatibility
#
# Cell 15–19 source_code / ast ব্যবহার করতে পারবে
# ============================================================

source_code = DEMO_SOURCE


# ============================================================
# ধাপ ১ — লেক্সার
# ============================================================

print("=" * 60)
print("CELL 14 — সম্পূর্ণ কম্পাইলার পরীক্ষা")
print("=" * 60)

print()
print("ধাপ ১ — লেক্সার")
print("-" * 60)


lexer = Lexer(source_code)

tokens = lexer.tokenize()


print("লেক্সার ত্রুটি:")


if lexer.errors:

    for error in lexer.errors:
        print(error)

else:

    print("কোনো লেক্সার ত্রুটি পাওয়া যায়নি! ✅")


# ============================================================
# টোকেনসমূহ
# ============================================================

print()
print("টোকেনসমূহ:")
print("-" * 60)


for token in tokens:
    print(token)


# ============================================================
# ধাপ ২ — পার্সার
# ============================================================

print()
print("=" * 60)
print("ধাপ ২ — পার্সার")
print("=" * 60)


parser = Parser(tokens)

ast = parser.parse()


# ============================================================
# ধাপ ৩ — AST
# ============================================================

print()
print("=" * 60)
print("ধাপ ৩ — তৈরি হওয়া AST")
print("=" * 60)

print()
print(ast)


# ============================================================
# ধাপ ৪ — পার্সার ত্রুটি
# ============================================================

print()
print("=" * 60)
print("ধাপ ৪ — পার্সার ত্রুটি")
print("=" * 60)


if parser.errors:

    print()

    for error in parser.errors:
        print(error)

else:

    print()
    print("কোনো পার্সার ত্রুটি পাওয়া যায়নি! ✅")


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 60)
print("CELL 14 — পরীক্ষা শেষ")
print("=" * 60)


print(
    '''
বাংলা পাইথন-স্টাইল কম্পাইলার পাইপলাইন:

বাংলা সোর্স কোড
       ↓
লেক্সার
       ↓
টোকেন
       ↓
পার্সার
       ↓
AST
'''
)


print("CELL 14 সফলভাবে সম্পন্ন হয়েছে! 🎉")

CELL 14 — সম্পূর্ণ কম্পাইলার পরীক্ষা

ধাপ ১ — লেক্সার
------------------------------------------------------------
লেক্সার ত্রুটি:
কোনো লেক্সার ত্রুটি পাওয়া যায়নি! ✅

টোকেনসমূহ:
------------------------------------------------------------
Token(TYPE_INT, 'পূর্ণসংখ্যা', line=6, column=1)
Token(IDENTIFIER, 'x', line=6, column=13)
Token(ASSIGN, '=', line=6, column=15)
Token(INTEGER, '10', line=6, column=17)
Token(NEWLINE, '', line=6, column=19)
Token(TYPE_FLOAT, 'দশমিক', line=7, column=1)
Token(IDENTIFIER, 'y', line=7, column=7)
Token(ASSIGN, '=', line=7, column=9)
Token(FLOAT, '2.5', line=7, column=11)
Token(NEWLINE, '', line=7, column=14)
Token(TYPE_STRING, 'স্ট্রিং', line=8, column=1)
Token(IDENTIFIER, 'নাম', line=8, column=9)
Token(ASSIGN, '=', line=8, column=13)
Token(STRING, 'বাংলাভাষা', line=8, column=15)
Token(NEWLINE, '', line=8, column=26)
Token(IDENTIFIER, 'x', line=15, column=1)
Token(ASSIGN, '=', line=15, column=3)
Token(IDENTIFIER, 'x', line=15, column=5)
Token(PLUS, '+', li

In [15]:
# ============================================================
# CELL 15 — SYMBOL TABLE
# Bangla Python-Style Compiler
# ============================================================


class Symbol:

    def __init__(
        self,
        name,
        symbol_type,
        data_type=None,
        params=None,
        properties=None
    ):

        self.name = name

        self.symbol_type = symbol_type

        self.data_type = data_type

        self.params = (
            params
            if params is not None
            else []
        )

        self.properties = (
            properties
            if properties is not None
            else {}
        )


    def __repr__(self):

        return (

            f"Symbol("
            f"name={self.name!r}, "
            f"symbol_type={self.symbol_type!r}, "
            f"data_type={self.data_type!r}, "
            f"params={self.params!r}, "
            f"properties={self.properties!r}"
            f")"

        )


# ============================================================
# SYMBOL TABLE
# ============================================================

class SymbolTable:


    def __init__(self):

        # Variables
        self.variables = {}


        # Functions
        self.functions = {}


        # Classes
        self.classes = {}


        # Stacks
        self.stacks = {}


        # Queues
        self.queues = {}


    # ========================================================
    # VARIABLES
    # ========================================================

    def define_variable(
        self,
        name,
        data_type
    ):

        symbol = Symbol(

            name,

            "VARIABLE",

            data_type=data_type

        )


        self.variables[name] = symbol


        return symbol


    def get_variable(
        self,
        name
    ):

        return self.variables.get(name)


    # ========================================================
    # FUNCTIONS
    # ========================================================

    def define_function(
        self,
        name,
        params
    ):

        symbol = Symbol(

            name,

            "FUNCTION",

            params=params

        )


        self.functions[name] = symbol


        return symbol


    def get_function(
        self,
        name
    ):

        return self.functions.get(name)


    # ========================================================
    # CLASSES
    # ========================================================

    def define_class(
        self,
        name,
        properties
    ):

        symbol = Symbol(

            name,

            "CLASS",

            properties=properties

        )


        self.classes[name] = symbol


        return symbol


    def get_class(
        self,
        name
    ):

        return self.classes.get(name)


    # ========================================================
    # STACK
    # ========================================================

    def define_stack(
        self,
        name
    ):

        symbol = Symbol(

            name,

            "STACK"

        )


        self.stacks[name] = symbol


        return symbol


    def get_stack(
        self,
        name
    ):

        return self.stacks.get(name)


    # ========================================================
    # QUEUE
    # ========================================================

    def define_queue(
        self,
        name
    ):

        symbol = Symbol(

            name,

            "QUEUE"

        )


        self.queues[name] = symbol


        return symbol


    def get_queue(
        self,
        name
    ):

        return self.queues.get(name)


    # ========================================================
    # GENERAL LOOKUP
    # ========================================================

    def lookup(
        self,
        name
    ):

        return (

            self.get_variable(name)

            or

            self.get_function(name)

            or

            self.get_class(name)

            or

            self.get_stack(name)

            or

            self.get_queue(name)

        )


    # ========================================================
    # DISPLAY
    # ========================================================

    def display(self):


        print(
            "\n" + "=" * 60
        )

        print(
            "SYMBOL TABLE"
        )

        print(
            "=" * 60
        )


        # ----------------------------------------------------
        # VARIABLES
        # ----------------------------------------------------

        print(
            "\nVARIABLES:"
        )


        if self.variables:


            for name, symbol in self.variables.items():

                print(

                    f"  {name} "
                    f"-> {symbol.data_type}"

                )


        else:

            print(
                "  None"
            )


        # ----------------------------------------------------
        # FUNCTIONS
        # ----------------------------------------------------

        print(
            "\nFUNCTIONS:"
        )


        if self.functions:


            for name, symbol in self.functions.items():

                print(

                    f"  {name} "
                    f"-> {symbol.params}"

                )


        else:

            print(
                "  None"
            )


        # ----------------------------------------------------
        # CLASSES
        # ----------------------------------------------------

        print(
            "\nCLASSES:"
        )


        if self.classes:


            for name, symbol in self.classes.items():

                print(

                    f"  {name} "
                    f"-> {symbol.properties}"

                )


        else:

            print(
                "  None"
            )


        # ----------------------------------------------------
        # STACKS
        # ----------------------------------------------------

        print(
            "\nSTACKS:"
        )


        if self.stacks:

            print(

                "  "

                +

                ", ".join(
                    self.stacks
                )

            )


        else:

            print(
                "  None"
            )


        # ----------------------------------------------------
        # QUEUES
        # ----------------------------------------------------

        print(
            "\nQUEUES:"
        )


        if self.queues:

            print(

                "  "

                +

                ", ".join(
                    self.queues
                )

            )


        else:

            print(
                "  None"
            )


        print(
            "\n" + "=" * 60
        )


# ============================================================
# IMPORTANT
#
# Keep the main Symbol Table EMPTY here.
#
# CELL 16 will fill it from the ACTUAL AST.
#
# This prevents the old Cell 15 test data from causing
# false "Duplicate declaration" errors.
# ============================================================


symbol_table = SymbolTable()


print(
    "CELL 15 — EMPTY SYMBOL TABLE READY! ✅"
)

CELL 15 — EMPTY SYMBOL TABLE READY! ✅


In [16]:
# ============================================================
# CELL 16 — SEMANTIC ANALYSIS
# Bangla Python-Style Compiler
#
# AST field compatibility:
#   FunctionCall.name
#   MemberAccess.object_expr
#
# FINAL PLAN checks:
# Duplicate variable
# Undefined variable
# Type mismatch
# Arithmetic compatibility
# Function parameter count/type
# Stack/Queue validation
# Class/property validation
# Return context
# ============================================================


TYPE_INT = "TYPE_INT"

TYPE_FLOAT = "TYPE_FLOAT"

TYPE_STRING = "TYPE_STRING"


# ============================================================
# SEMANTIC ANALYZER
# ============================================================

class SemanticAnalyzer:


    # ========================================================
    # INITIALIZATION
    # ========================================================

    def __init__(
        self,
        symbol_table
    ):

        self.symbol_table = symbol_table

        self.errors = []

        # Scope stack
        self.scopes = [{}]

        # Functions
        self.functions = {}

        # Classes
        self.classes = {}

        # Stack names
        self.stacks = set()

        # Queue names
        self.queues = set()

        # Return validation
        self.function_depth = 0


    # ========================================================
    # ERROR
    # ========================================================

    def error(
        self,
        message
    ):

        self.errors.append(

            f"Semantic Error: {message}"

        )


    # ========================================================
    # ENTER SCOPE
    # ========================================================

    def enter_scope(
        self
    ):

        self.scopes.append(
            {}
        )


    # ========================================================
    # EXIT SCOPE
    # ========================================================

    def exit_scope(
        self
    ):

        if len(self.scopes) > 1:

            self.scopes.pop()


    # ========================================================
    # NORMALIZE TYPE
    # ========================================================

    def normalize_type(
        self,
        data_type
    ):

        mapping = {

            "পূর্ণসংখ্যা": TYPE_INT,

            "দশমিক": TYPE_FLOAT,

            "স্ট্রিং": TYPE_STRING,

            "int": TYPE_INT,

            "float": TYPE_FLOAT,

            "str": TYPE_STRING,

            TYPE_INT: TYPE_INT,

            TYPE_FLOAT: TYPE_FLOAT,

            TYPE_STRING: TYPE_STRING

        }


        return mapping.get(
            data_type,
            data_type
        )


    # ========================================================
    # LOOKUP VARIABLE
    # ========================================================

    def lookup_variable(
        self,
        name
    ):

        for scope in reversed(
            self.scopes
        ):

            if name in scope:

                return scope[
                    name
                ]


        return None


    # ========================================================
    # DECLARE VARIABLE
    # ========================================================

    def declare_variable(
        self,
        name,
        data_type
    ):

        current_scope = (
            self.scopes[-1]
        )


        if name in current_scope:

            self.error(

                f"Duplicate declaration "
                f"of '{name}'"

            )

            return False


        current_scope[
            name
        ] = data_type


        return True


    # ========================================================
    # TYPE COMPATIBILITY
    # ========================================================

    def types_compatible(
        self,
        expected,
        actual
    ):

        if expected is None:

            return True


        if actual is None:

            return True


        expected = (
            self.normalize_type(
                expected
            )
        )


        actual = (
            self.normalize_type(
                actual
            )
        )


        if expected == actual:

            return True


        # পূর্ণসংখ্যা float এ ব্যবহার করা যাবে

        if (

            expected == TYPE_FLOAT

            and

            actual == TYPE_INT

        ):

            return True


        return False


    # ========================================================
    # GENERIC VISITOR
    # ========================================================

    def visit(
        self,
        node
    ):

        if node is None:

            return None


        method_name = (

            f"visit_{type(node).__name__}"

        )


        method = getattr(

            self,

            method_name,

            self.visit_unknown

        )


        return method(
            node
        )


    # ========================================================
    # UNKNOWN NODE
    # ========================================================

    def visit_unknown(
        self,
        node
    ):

        self.error(

            f"Unsupported AST node "
            f"'{type(node).__name__}'"

        )


        return None


    # ========================================================
    # MAIN ANALYZE
    # ========================================================

    def analyze(
        self,
        program
    ):


        # Reset

        self.errors = []

        self.scopes = [{}]

        self.functions = {}

        self.classes = {}

        self.stacks = set()

        self.queues = set()

        self.function_depth = 0


        # ----------------------------------------------------
        # Clear Symbol Table
        # ----------------------------------------------------

        self.symbol_table.variables.clear()

        self.symbol_table.functions.clear()

        self.symbol_table.classes.clear()

        self.symbol_table.stacks.clear()

        self.symbol_table.queues.clear()


        # ----------------------------------------------------
        # PROGRAM
        # ----------------------------------------------------

        if isinstance(
            program,
            Program
        ):


            # =================================================
            # FIRST PASS
            #
            # Collect functions and classes
            # =================================================

            for statement in (
                program.statements
            ):


                if isinstance(

                    statement,

                    FunctionDecl

                ):

                    self.collect_function(
                        statement
                    )


                elif isinstance(

                    statement,

                    ClassDecl

                ):

                    self.collect_class(
                        statement
                    )


            # =================================================
            # SECOND PASS
            #
            # Full semantic analysis
            # =================================================

            for statement in (
                program.statements
            ):

                self.visit(
                    statement
                )


        else:

            self.visit(
                program
            )


        return self.errors


    # ========================================================
    # NUMBER
    # ========================================================

    def visit_Number(
        self,
        node
    ):


        if isinstance(
            node.value,
            float
        ):

            return TYPE_FLOAT


        return TYPE_INT


    # ========================================================
    # STRING
    # ========================================================

    def visit_String(
        self,
        node
    ):

        return TYPE_STRING


    # ========================================================
    # VARIABLE
    # ========================================================

    def visit_Variable(
        self,
        node
    ):


        data_type = (

            self.lookup_variable(
                node.name
            )

        )


        if data_type is None:

            self.error(

                f"Undefined variable "
                f"'{node.name}'"

            )


            return None


        return data_type


    # ========================================================
    # UNARY OPERATION
    # ========================================================

    def visit_UnaryOp(
        self,
        node
    ):


        operand_type = (

            self.visit(
                node.operand
            )

        )


        if (

            node.operator == "-"

            and

            operand_type not in (

                TYPE_INT,

                TYPE_FLOAT,

                None

            )

        ):

            self.error(

                "Unary '-' requires "
                "a numeric value"

            )


        return operand_type


    # ========================================================
    # BINARY OPERATION
    # ========================================================

    def visit_BinaryOp(
        self,
        node
    ):


        left_type = (

            self.visit(
                node.left
            )

        )


        right_type = (

            self.visit(
                node.right
            )

        )


        operator = (
            node.operator
        )


        # ====================================================
        # ARITHMETIC
        # ====================================================

        if operator in (

            "+",

            "-",

            "*",

            "/"

        ):


            if (

                left_type not in (

                    TYPE_INT,

                    TYPE_FLOAT,

                    None

                )

                or

                right_type not in (

                    TYPE_INT,

                    TYPE_FLOAT,

                    None

                )

            ):


                self.error(

                    f"Arithmetic operator "
                    f"'{operator}' "
                    f"requires numeric operands"

                )


                return None


            if operator == "/":

                return TYPE_FLOAT


            if (

                left_type == TYPE_FLOAT

                or

                right_type == TYPE_FLOAT

            ):

                return TYPE_FLOAT


            return TYPE_INT


        # ====================================================
        # COMPARISON
        # ====================================================

        if operator in (

            ">",

            "<",

            ">=",

            "<=",

            "==",

            "!="

        ):


            if (

                left_type is not None

                and

                right_type is not None

            ):


                numeric_pair = (

                    {

                        left_type,

                        right_type

                    }

                    <=

                    {

                        TYPE_INT,

                        TYPE_FLOAT

                    }

                )


                if (

                    not numeric_pair

                    and

                    left_type != right_type

                ):

                    self.error(

                        f"Incompatible types "
                        f"for comparison "
                        f"'{operator}'"

                    )


            # Source language-এ Boolean type নেই

            return "COMPARISON"


        self.error(

            f"Unsupported operator "
            f"'{operator}'"

        )


        return None


    # ========================================================
    # PROGRAM
    # ========================================================

    def visit_Program(
        self,
        node
    ):

        return self.analyze(
            node
        )


    # ========================================================
    # BLOCK
    # ========================================================

    def visit_Block(
        self,
        node
    ):


        self.enter_scope()


        try:


            for statement in (

                node.statements

            ):

                self.visit(
                    statement
                )


        finally:

            self.exit_scope()


    # ========================================================
    # VARIABLE DECLARATION
    # ========================================================

    def visit_VarDecl(
        self,
        node
    ):


        declared_type = (

            self.normalize_type(
                node.var_type
            )

        )


        # ----------------------------------------------------
        # Value type
        # ----------------------------------------------------

        if node.value is not None:

            value_type = (

                self.visit(
                    node.value
                )

            )

        else:

            value_type = None


        # ----------------------------------------------------
        # Valid type
        # ----------------------------------------------------

        if (

            declared_type not in (

                TYPE_INT,

                TYPE_FLOAT,

                TYPE_STRING

            )

            and

            declared_type not in self.classes

        ):

            self.error(

                f"Unknown type "
                f"'{node.var_type}'"

            )


        # ----------------------------------------------------
        # Type mismatch
        # ----------------------------------------------------

        if (

            value_type is not None

            and

            value_type != "COMPARISON"

        ):


            if not self.types_compatible(

                declared_type,

                value_type

            ):

                self.error(

                    f"Type mismatch for "
                    f"'{node.name}': "
                    f"expected '{declared_type}', "
                    f"got '{value_type}'"

                )


        # ----------------------------------------------------
        # Declare
        # ----------------------------------------------------

        if self.declare_variable(

            node.name,

            declared_type

        ):


            # Top-level variable only

            if len(self.scopes) == 1:

                self.symbol_table.define_variable(

                    node.name,

                    declared_type

                )


        return declared_type


    # ========================================================
    # ASSIGNMENT
    # ========================================================

    def visit_Assignment(
        self,
        node
    ):


        value_type = (

            self.visit(
                node.value
            )

        )


        # ====================================================
        # NORMAL VARIABLE
        # ====================================================

        if isinstance(

            node.target,

            Variable

        ):


            expected_type = (

                self.lookup_variable(
                    node.target.name
                )

            )


            if expected_type is None:

                self.error(

                    f"Undefined variable "
                    f"'{node.target.name}'"

                )


                return None


            if (

                value_type is not None

                and

                value_type != "COMPARISON"

            ):


                if not self.types_compatible(

                    expected_type,

                    value_type

                ):

                    self.error(

                        f"Type mismatch assigning "
                        f"to '{node.target.name}': "
                        f"expected '{expected_type}', "
                        f"got '{value_type}'"

                    )


            return expected_type


        # ====================================================
        # OBJECT PROPERTY
        # ====================================================

        if isinstance(

            node.target,

            MemberAccess

        ):


            expected_type = (

                self.get_member_type(
                    node.target
                )

            )


            if (

                expected_type is not None

                and

                value_type is not None

            ):


                if not self.types_compatible(

                    expected_type,

                    value_type

                ):

                    self.error(

                        f"Type mismatch assigning "
                        f"property "
                        f"'{node.target.member}'"

                    )


            return expected_type


        self.error(

            "Invalid assignment target"

        )


        return None


    # ========================================================
    # PRINT
    # ========================================================

    def visit_PrintStatement(
        self,
        node
    ):

        return self.visit(
            node.expression
        )


    # ========================================================
    # EXPRESSION STATEMENT
    # ========================================================

    def visit_ExpressionStatement(
        self,
        node
    ):

        return self.visit(
            node.expression
        )


    # ========================================================
    # IF
    # ========================================================

    def visit_IfStatement(
        self,
        node
    ):


        self.visit(
            node.condition
        )


        self.visit(
            node.then_branch
        )


        if node.else_branch is not None:

            self.visit(
                node.else_branch
            )


    # ========================================================
    # WHILE
    # ========================================================

    def visit_WhileStatement(
        self,
        node
    ):


        self.visit(
            node.condition
        )


        self.visit(
            node.body
        )


    # ========================================================
    # COLLECT FUNCTION
    # ========================================================

    def collect_function(
        self,
        node
    ):


        if node.name in self.functions:

            self.error(

                f"Duplicate declaration "
                f"of function "
                f"'{node.name}'"

            )


            return


        params = []

        seen = set()


        for param_type, param_name in (

            node.parameters

        ):


            normalized = (

                self.normalize_type(
                    param_type
                )

            )


            if normalized not in (

                TYPE_INT,

                TYPE_FLOAT,

                TYPE_STRING

            ):

                self.error(

                    f"Invalid parameter type "
                    f"'{param_type}'"

                )


            if param_name in seen:

                self.error(

                    f"Duplicate parameter "
                    f"'{param_name}' "
                    f"in function "
                    f"'{node.name}'"

                )


            seen.add(
                param_name
            )


            params.append(

                (

                    normalized,

                    param_name

                )

            )


        self.functions[
            node.name
        ] = {

            "params": params

        }


        self.symbol_table.define_function(

            node.name,

            params

        )


    # ========================================================
    # FUNCTION DECLARATION
    # ========================================================

    def visit_FunctionDecl(
        self,
        node
    ):


        if node.name not in self.functions:

            self.collect_function(
                node
            )


        info = (

            self.functions.get(
                node.name
            )

        )


        if info is None:

            return


        self.enter_scope()

        self.function_depth += 1


        try:


            # Parameters

            for param_type, param_name in (

                info["params"]

            ):

                self.declare_variable(

                    param_name,

                    param_type

                )


            # Function body

            for statement in (

                node.body.statements

            ):

                self.visit(
                    statement
                )


        finally:

            self.function_depth -= 1

            self.exit_scope()


    # ========================================================
    # RETURN
    # ========================================================

    def visit_ReturnStatement(
        self,
        node
    ):


        if self.function_depth <= 0:

            self.error(

                "'ফেরত' শুধুমাত্র "
                "ফাংশনের ভিতরে ব্যবহার করা যাবে"

            )


        return self.visit(
            node.value
        )


    # ========================================================
    # FUNCTION CALL
    # ========================================================

    def visit_FunctionCall(
        self,
        node
    ):


        # ====================================================
        # IMPORTANT
        #
        # Current AST:
        #
        # FunctionCall.name
        #
        # NOT:
        #
        # FunctionCall.callee
        # ====================================================

        callee = (
            node.name
        )


        # ====================================================
        # MEMBER CALL
        #
        # s.ঠেলো(...)
        # q.সামনে()
        # ====================================================

        if isinstance(

            callee,

            MemberAccess

        ):

            return self.visit_member_call(

                callee,

                node.arguments

            )


        # ====================================================
        # NORMAL FUNCTION
        # ====================================================

        if isinstance(

            callee,

            Variable

        ):

            function_name = (
                callee.name
            )


        else:

            function_name = (
                callee
            )


        if not isinstance(

            function_name,

            str

        ):

            self.error(

                "Invalid function "
                "call target"

            )


            return None


        info = (

            self.functions.get(
                function_name
            )

        )


        if info is None:

            self.error(

                f"Undefined function "
                f"'{function_name}'"

            )


            return None


        params = (

            info["params"]

        )


        # ====================================================
        # PARAMETER COUNT
        # ====================================================

        if (

            len(node.arguments)

            !=

            len(params)

        ):

            self.error(

                f"Function "
                f"'{function_name}' "
                f"expects "
                f"{len(params)} "
                f"argument(s), "
                f"but got "
                f"{len(node.arguments)}"

            )


        # ====================================================
        # PARAMETER TYPE
        # ====================================================

        for index, argument in enumerate(

            node.arguments

        ):


            actual_type = (

                self.visit(
                    argument
                )

            )


            if index < len(params):


                expected_type = (

                    params[index][0]

                )


                if not self.types_compatible(

                    expected_type,

                    actual_type

                ):

                    self.error(

                        f"Function "
                        f"'{function_name}' "
                        f"argument "
                        f"{index + 1} "
                        f"type mismatch: "
                        f"expected "
                        f"'{expected_type}', "
                        f"got "
                        f"'{actual_type}'"

                    )


        # Final plan-এ function return type
        # declaration syntax নেই

        return None


    # ========================================================
    # STACK DECLARATION
    # ========================================================

    def visit_StackDecl(
        self,
        node
    ):


        name = node.name


        if (

            name in self.stacks

            or

            name in self.queues

            or

            self.lookup_variable(
                name
            )
            is not None

        ):

            self.error(

                f"Duplicate declaration "
                f"of '{name}'"

            )


            return


        self.stacks.add(
            name
        )


        self.symbol_table.define_stack(
            name
        )


    # ========================================================
    # QUEUE DECLARATION
    # ========================================================

    def visit_QueueDecl(
        self,
        node
    ):


        name = node.name


        if (

            name in self.queues

            or

            name in self.stacks

            or

            self.lookup_variable(
                name
            )
            is not None

        ):

            self.error(

                f"Duplicate declaration "
                f"of '{name}'"

            )


            return


        self.queues.add(
            name
        )


        self.symbol_table.define_queue(
            name
        )


    # ========================================================
    # MEMBER CALL
    # ========================================================

    def visit_member_call(
        self,
        callee,
        arguments
    ):


        # ====================================================
        # IMPORTANT
        #
        # Current AST:
        #
        # MemberAccess.object_expr
        #
        # NOT:
        #
        # MemberAccess.object
        # ====================================================

        object_node = (

            callee.object_expr

        )


        member = (

            callee.member

        )


        if not isinstance(

            object_node,

            Variable

        ):

            self.error(

                "Invalid member "
                "call target"

            )


            return None


        object_name = (

            object_node.name

        )


        # ====================================================
        # STACK
        # ====================================================

        if object_name in self.stacks:


            operations = {

                "ঠেলো": 1,

                "বের_করো": 0,

                "উপর_দেখো": 0

            }


            if member not in operations:

                self.error(

                    f"Invalid Stack operation "
                    f"'{member}'"

                )


            elif (

                len(arguments)

                !=

                operations[member]

            ):

                self.error(

                    f"Stack operation "
                    f"'{member}' "
                    f"expects "
                    f"{operations[member]} "
                    f"argument(s)"

                )


            for argument in arguments:

                self.visit(
                    argument
                )


            return None


        # ====================================================
        # QUEUE
        # ====================================================

        if object_name in self.queues:


            operations = {

                "ঢোকাও": 1,

                "বের_করো": 0,

                "সামনে": 0

            }


            if member not in operations:

                self.error(

                    f"Invalid Queue operation "
                    f"'{member}'"

                )


            elif (

                len(arguments)

                !=

                operations[member]

            ):

                self.error(

                    f"Queue operation "
                    f"'{member}' "
                    f"expects "
                    f"{operations[member]} "
                    f"argument(s)"

                )


            for argument in arguments:

                self.visit(
                    argument
                )


            return None


        self.error(

            f"'{object_name}.{member}' "
            f"is not a valid "
            f"Stack or Queue operation"

        )


        return None


    # ========================================================
    # COLLECT CLASS
    # ========================================================

    def collect_class(
        self,
        node
    ):


        if node.name in self.classes:

            self.error(

                f"Duplicate declaration "
                f"of class "
                f"'{node.name}'"

            )


            return


        properties = {}


        for property_node in (

            node.properties

        ):


            if not isinstance(

                property_node,

                VarDecl

            ):

                self.error(

                    f"Invalid property "
                    f"inside class "
                    f"'{node.name}'"

                )


                continue


            property_type = (

                self.normalize_type(

                    property_node.var_type

                )

            )


            if (

                property_node.name

                in

                properties

            ):

                self.error(

                    f"Duplicate property "
                    f"'{property_node.name}' "
                    f"in class "
                    f"'{node.name}'"

                )


                continue


            properties[
                property_node.name
            ] = property_type


        self.classes[
            node.name
        ] = properties


        self.symbol_table.define_class(

            node.name,

            properties

        )


    # ========================================================
    # CLASS DECLARATION
    # ========================================================

    def visit_ClassDecl(
        self,
        node
    ):


        if node.name not in self.classes:

            self.collect_class(
                node
            )


        properties = (

            self.classes.get(

                node.name,

                {}

            )

        )


        # ----------------------------------------------------
        # Property initializer type check
        # ----------------------------------------------------

        for property_node in (

            node.properties

        ):


            if not isinstance(

                property_node,

                VarDecl

            ):

                continue


            if property_node.value is None:

                continue


            expected_type = (

                properties.get(

                    property_node.name

                )

            )


            actual_type = (

                self.visit(

                    property_node.value

                )

            )


            if (

                expected_type is not None

                and

                not self.types_compatible(

                    expected_type,

                    actual_type

                )

            ):

                self.error(

                    f"Type mismatch in property "
                    f"'{property_node.name}' "
                    f"of class "
                    f"'{node.name}'"

                )


    # ========================================================
    # MEMBER TYPE
    # ========================================================

    def get_member_type(
        self,
        node,
        report_error=True
    ):


        # Current AST field

        object_node = (

            node.object_expr

        )


        member = (

            node.member

        )


        if not isinstance(

            object_node,

            Variable

        ):


            if report_error:

                self.error(

                    "Invalid member access"

                )


            return None


        object_name = (

            object_node.name

        )


        object_type = (

            self.lookup_variable(

                object_name

            )

        )


        if object_type is None:


            if report_error:

                self.error(

                    f"Undefined variable "
                    f"'{object_name}'"

                )


            return None


        # ====================================================
        # CLASS CHECK
        # ====================================================

        if object_type not in self.classes:


            if report_error:

                self.error(

                    f"Cannot access property "
                    f"'{member}' "
                    f"from non-object type "
                    f"'{object_type}'"

                )


            return None


        properties = (

            self.classes[
                object_type
            ]

        )


        # ====================================================
        # PROPERTY EXISTS
        # ====================================================

        if member not in properties:


            if report_error:

                self.error(

                    f"Class "
                    f"'{object_type}' "
                    f"has no property "
                    f"'{member}'"

                )


            return None


        return properties[
            member
        ]


    # ========================================================
    # MEMBER ACCESS
    # ========================================================

    def visit_MemberAccess(
        self,
        node
    ):

        return self.get_member_type(

            node,

            report_error=True

        )


    # ========================================================
    # NEW OBJECT
    # ========================================================

    def visit_NewObject(
        self,
        node
    ):


        if node.class_name not in self.classes:

            self.error(

                f"Undefined class "
                f"'{node.class_name}'"

            )


            return None


        return node.class_name


# ============================================================
# CELL 16 TEST
# ============================================================

print("=" * 60)

print(
    "CELL 16 — SEMANTIC ANALYSIS"
)

print("=" * 60)


# ============================================================
# CREATE ANALYZER
# ============================================================

semantic_analyzer = SemanticAnalyzer(
    symbol_table
)


# ============================================================
# ANALYZE AST
# ============================================================

semantic_errors = (

    semantic_analyzer.analyze(
        ast
    )

)


# ============================================================
# DISPLAY ERRORS
# ============================================================

if semantic_errors:


    print(
        "\nSEMANTIC ERRORS FOUND:\n"
    )


    for error in semantic_errors:

        print(
            error
        )


else:

    print(
        "\nকোনো Semantic Error পাওয়া যায়নি! ✅"
    )


# ============================================================
# DISPLAY SYMBOL TABLE
# ============================================================

print(
    "\n" + "=" * 60
)

print(
    "SYMBOL TABLE AFTER SEMANTIC ANALYSIS"
)

print(
    "=" * 60
)


symbol_table.display()


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n" + "=" * 60
)

print(
    "SEMANTIC ANALYSIS SUMMARY"
)

print(
    "=" * 60
)


print(
    "\n✓ Duplicate declaration check"
)

print(
    "✓ Undefined variable check"
)

print(
    "✓ Declaration type mismatch"
)

print(
    "✓ Assignment type mismatch"
)

print(
    "✓ Arithmetic type compatibility"
)

print(
    "✓ Function parameter count/type"
)

print(
    "✓ Stack operation validation"
)

print(
    "✓ Queue operation validation"
)

print(
    "✓ Class existence"
)

print(
    "✓ Property existence"
)

print(
    "✓ Return statement context check"
)


print(
    "\nCELL 16 — SEMANTIC ANALYSIS FINISHED! 🎉"
)

CELL 16 — SEMANTIC ANALYSIS

কোনো Semantic Error পাওয়া যায়নি! ✅

SYMBOL TABLE AFTER SEMANTIC ANALYSIS

SYMBOL TABLE

VARIABLES:
  x -> TYPE_INT
  y -> TYPE_FLOAT
  নাম -> TYPE_STRING
  ফল -> TYPE_INT
  ছাত্র -> শিক্ষার্থী

FUNCTIONS:
  যোগ -> [('TYPE_INT', 'a'), ('TYPE_INT', 'b')]

CLASSES:
  শিক্ষার্থী -> {'নাম': 'TYPE_STRING', 'বয়স': 'TYPE_INT'}

STACKS:
  s

QUEUES:
  q


SEMANTIC ANALYSIS SUMMARY

✓ Duplicate declaration check
✓ Undefined variable check
✓ Declaration type mismatch
✓ Assignment type mismatch
✓ Arithmetic type compatibility
✓ Function parameter count/type
✓ Stack operation validation
✓ Queue operation validation
✓ Class existence
✓ Property existence
✓ Return statement context check

CELL 16 — SEMANTIC ANALYSIS FINISHED! 🎉


In [17]:
# ============================================================
# CELL 17 — THREE ADDRESS CODE (TAC) GENERATOR
# Bangla Python-Style Compiler
#
# FINAL PLAN অনুযায়ী:
# Arithmetic
# Assignment
# IF–ELSE
# WHILE
# ============================================================


class TACGenerator:


    # ========================================================
    # INITIALIZATION
    # ========================================================

    def __init__(self):

        self.code = []

        self.temp_count = 0

        self.label_count = 0


    # ========================================================
    # NEW TEMPORARY
    # ========================================================

    def new_temp(self):

        self.temp_count += 1

        return f"t{self.temp_count}"


    # ========================================================
    # NEW LABEL
    # ========================================================

    def new_label(self):

        self.label_count += 1

        return f"L{self.label_count}"


    # ========================================================
    # ADD TAC INSTRUCTION
    # ========================================================

    def emit(self, instruction):

        self.code.append(instruction)


    # ========================================================
    # MAIN GENERATE
    # ========================================================

    def generate(self, node):


        # ====================================================
        # EMPTY
        # ====================================================

        if node is None:

            return None


        # ====================================================
        # PROGRAM
        # ====================================================

        if isinstance(node, Program):

            for statement in node.statements:

                self.generate(statement)

            return None


        # ====================================================
        # BLOCK
        # ====================================================

        if isinstance(node, Block):

            for statement in node.statements:

                self.generate(statement)

            return None


        # ====================================================
        # VARIABLE DECLARATION
        #
        # পূর্ণসংখ্যা x = 10
        # ====================================================

        if isinstance(node, VarDecl):

            value = self.generate(node.value)

            self.emit(
                f"{node.name} = {value}"
            )

            return node.name


        # ====================================================
        # ASSIGNMENT
        #
        # x = x + 5
        #
        # ছাত্র.নাম = "রহিম"
        # ====================================================

        if isinstance(node, Assignment):

            value = self.generate(
                node.value
            )


            # Normal variable

            if isinstance(
                node.target,
                Variable
            ):

                target = node.target.name


            # Object property

            elif isinstance(
                node.target,
                MemberAccess
            ):

                object_name = self.generate(
                    node.target.object_expr
                )

                target = (
                    f"{object_name}."
                    f"{node.target.member}"
                )


            else:

                target = str(
                    node.target
                )


            self.emit(
                f"{target} = {value}"
            )

            return target


        # ====================================================
        # PRINT
        #
        # লেখো(x)
        # ====================================================

        if isinstance(
            node,
            PrintStatement
        ):

            value = self.generate(
                node.expression
            )

            self.emit(
                f"PRINT {value}"
            )

            return None


        # ====================================================
        # EXPRESSION STATEMENT
        #
        # s.ঠেলো(100)
        # q.ঢোকাও("ক")
        # ====================================================

        if isinstance(
            node,
            ExpressionStatement
        ):

            self.generate(
                node.expression
            )

            return None


        # ====================================================
        # IF – ELSE
        #
        # যদি x > 5:
        #     লেখো("বড়")
        # নাহলে:
        #     লেখো("ছোট")
        # ====================================================

        if isinstance(
            node,
            IfStatement
        ):


            # Condition

            condition = self.generate(
                node.condition
            )


            # Labels

            else_label = self.new_label()

            end_label = self.new_label()


            # Condition false হলে ELSE

            self.emit(
                f"ifFalse {condition} "
                f"goto {else_label}"
            )


            # -----------------------------------------------
            # IF BODY
            # -----------------------------------------------

            self.generate(
                node.then_branch
            )


            # ELSE থাকলে END এ jump

            if node.else_branch is not None:

                self.emit(
                    f"goto {end_label}"
                )


            # -----------------------------------------------
            # ELSE LABEL
            # -----------------------------------------------

            self.emit(
                f"{else_label}:"
            )


            # -----------------------------------------------
            # ELSE BODY
            # -----------------------------------------------

            if node.else_branch is not None:

                self.generate(
                    node.else_branch
                )


                self.emit(
                    f"{end_label}:"
                )


            return None


        # ====================================================
        # WHILE
        #
        # যতক্ষণ x < 10:
        #     লেখো(x)
        #     x = x + 1
        # ====================================================

        if isinstance(
            node,
            WhileStatement
        ):


            start_label = self.new_label()

            end_label = self.new_label()


            # -----------------------------------------------
            # LOOP START
            # -----------------------------------------------

            self.emit(
                f"{start_label}:"
            )


            # Condition

            condition = self.generate(
                node.condition
            )


            # False হলে loop শেষ

            self.emit(
                f"ifFalse {condition} "
                f"goto {end_label}"
            )


            # Body

            self.generate(
                node.body
            )


            # আবার শুরুতে

            self.emit(
                f"goto {start_label}"
            )


            # End

            self.emit(
                f"{end_label}:"
            )


            return None


        # ====================================================
        # FUNCTION
        # ====================================================

        if isinstance(
            node,
            FunctionDecl
        ):


            self.emit(
                f"FUNC {node.name}:"
            )


            self.generate(
                node.body
            )


            self.emit(
                f"END FUNC {node.name}"
            )


            return None


        # ====================================================
        # RETURN
        # ====================================================

        if isinstance(
            node,
            ReturnStatement
        ):

            value = self.generate(
                node.value
            )


            self.emit(
                f"RETURN {value}"
            )


            return None


        # ====================================================
        # STACK
        # ====================================================

        if isinstance(
            node,
            StackDecl
        ):

            self.emit(
                f"{node.name} = STACK"
            )

            return node.name


        # ====================================================
        # QUEUE
        # ====================================================

        if isinstance(
            node,
            QueueDecl
        ):

            self.emit(
                f"{node.name} = QUEUE"
            )

            return node.name


        # ====================================================
        # CLASS
        # ====================================================

        if isinstance(
            node,
            ClassDecl
        ):


            self.emit(
                f"CLASS {node.name}:"
            )


            for property_node in node.properties:


                if isinstance(
                    property_node,
                    VarDecl
                ):

                    value = self.generate(
                        property_node.value
                    )


                    self.emit(
                        f"  {property_node.name} "
                        f"= {value}"
                    )


            self.emit(
                f"END CLASS {node.name}"
            )


            return None


        # ====================================================
        # NUMBER
        # ====================================================

        if isinstance(
            node,
            Number
        ):

            return str(
                node.value
            )


        # ====================================================
        # STRING
        # ====================================================

        if isinstance(
            node,
            String
        ):

            return repr(
                node.value
            )


        # ====================================================
        # VARIABLE
        # ====================================================

        if isinstance(
            node,
            Variable
        ):

            return node.name


        # ====================================================
        # UNARY
        #
        # -x
        # ====================================================

        if isinstance(
            node,
            UnaryOp
        ):


            operand = self.generate(
                node.operand
            )


            temp = self.new_temp()


            self.emit(
                f"{temp} = "
                f"{node.operator}{operand}"
            )


            return temp


        # ====================================================
        # BINARY
        #
        # a + b
        # ====================================================

        if isinstance(
            node,
            BinaryOp
        ):


            left = self.generate(
                node.left
            )


            right = self.generate(
                node.right
            )


            temp = self.new_temp()


            self.emit(
                f"{temp} = "
                f"{left} "
                f"{node.operator} "
                f"{right}"
            )


            return temp


        # ====================================================
        # FUNCTION CALL
        #
        # যোগ(10, 20)
        #
        # s.ঠেলো(100)
        #
        # q.সামনে()
        # ====================================================

        if isinstance(
            node,
            FunctionCall
        ):


            arguments = []


            for argument in node.arguments:

                arguments.append(
                    self.generate(
                        argument
                    )
                )


            arguments_text = ", ".join(
                arguments
            )


            # -----------------------------------------------
            # MEMBER CALL
            #
            # s.ঠেলো(...)
            # s.উপর_দেখো()
            # q.ঢোকাও(...)
            # -----------------------------------------------

            if isinstance(
                node.name,
                MemberAccess
            ):


                receiver = self.generate(
                    node.name.object_expr
                )


                member_name = (
                    node.name.member
                )


                temp = self.new_temp()


                self.emit(
                    f"{temp} = CALL "
                    f"{receiver}."
                    f"{member_name}"
                    f"({arguments_text})"
                )


                return temp


            # -----------------------------------------------
            # NORMAL FUNCTION CALL
            # -----------------------------------------------

            if isinstance(
                node.name,
                str
            ):

                function_name = (
                    node.name
                )


            else:

                function_name = self.generate(
                    node.name
                )


            temp = self.new_temp()


            self.emit(
                f"{temp} = CALL "
                f"{function_name}"
                f"({arguments_text})"
            )


            return temp


        # ====================================================
        # MEMBER ACCESS
        #
        # ছাত্র.নাম
        # ====================================================

        if isinstance(
            node,
            MemberAccess
        ):


            receiver = self.generate(
                node.object_expr
            )


            return (
                f"{receiver}."
                f"{node.member}"
            )


        # ====================================================
        # NEW OBJECT
        #
        # নতুন শিক্ষার্থী()
        # ====================================================

        if isinstance(
            node,
            NewObject
        ):


            temp = self.new_temp()


            self.emit(
                f"{temp} = NEW "
                f"{node.class_name}"
            )


            return temp


        # ====================================================
        # FALLBACK
        # ====================================================

        return str(node)


    # ========================================================
    # GET TAC CODE
    # ========================================================

    def get_code(self):

        return self.code


    # ========================================================
    # DISPLAY
    # ========================================================

    def display(self):


        print()

        print("=" * 60)

        print("থ্রি অ্যাড্রেস কোড (TAC)")

        print("=" * 60)


        if not self.code:

            print(
                "কোনো TAC Instruction তৈরি হয়নি।"
            )


        else:

            for instruction in self.code:

                print(
                    instruction
                )


        print("=" * 60)


# ============================================================
# CELL 17 TEST
# ============================================================


print()

print("=" * 60)

print("CELL 17 — TAC তৈরি")

print("=" * 60)


# ============================================================
# CURRENT AST খোঁজা
# ============================================================


current_ast = None


possible_ast_names = [

    "ast",

    "program_ast",

    "generated_ast",

    "parser_ast"

]


for variable_name in possible_ast_names:


    if variable_name in globals():


        current_ast = globals()[
            variable_name
        ]


        break


# ============================================================
# AST পাওয়া না গেলে
# ============================================================


if current_ast is None:


    print()

    print(
        "⚠️ Program AST পাওয়া যায়নি।"
    )

    print(
        "আগে Cell 14 run করুন।"
    )


# ============================================================
# TAC GENERATION
# ============================================================

else:


    tac_generator = TACGenerator()


    tac_generator.generate(
        current_ast
    )


    tac_code = (
        tac_generator.get_code()
    )


    tac_generator.display()


    print()

    print(
        "মোট TAC Instruction:",
        len(tac_code)
    )


print()

print(
    "CELL 17 — TAC তৈরি শেষ! 🎉"
)


CELL 17 — TAC তৈরি

থ্রি অ্যাড্রেস কোড (TAC)
x = 10
y = 2.5
নাম = 'বাংলাভাষা'
t1 = 5 * 2
t2 = x + t1
x = t2
PRINT 'বাংলা কম্পাইলারে স্বাগতম'
PRINT নাম
PRINT x
t3 = x > 5
ifFalse t3 goto L1
PRINT 'মান বড়'
goto L2
L1:
PRINT 'মান ছোট'
L2:
L3:
t4 = x < 25
ifFalse t4 goto L4
PRINT x
t5 = x + 1
x = t5
goto L3
L4:
FUNC যোগ:
t6 = a + b
RETURN t6
END FUNC যোগ
t7 = CALL যোগ(10, 20)
ফল = t7
PRINT ফল
s = STACK
t8 = CALL s.ঠেলো(100)
t9 = CALL s.ঠেলো(200)
t10 = CALL s.উপর_দেখো()
PRINT t10
t11 = CALL s.বের_করো()
PRINT t11
q = QUEUE
t12 = CALL q.ঢোকাও('ক')
t13 = CALL q.ঢোকাও('খ')
t14 = CALL q.সামনে()
PRINT t14
t15 = CALL q.বের_করো()
PRINT t15
CLASS শিক্ষার্থী:
  নাম = 'অজানা'
  বয়স = 0
END CLASS শিক্ষার্থী
t16 = NEW শিক্ষার্থী
ছাত্র = t16
ছাত্র.নাম = 'রহিম'
PRINT ছাত্র.নাম

মোট TAC Instruction: 53

CELL 17 — TAC তৈরি শেষ! 🎉


In [18]:
# ============================================================
# CELL 18 — PYTHON CODE GENERATOR
# Bangla Python-Style Compiler
#
# AST
#   ↓
# Python Backend Code
#
# Source Language : Bangla
# Source Syntax   : Python-style indentation
# Target Language : Python
#
# User লিখবে:
#
#     যদি x > 5:
#         লেখো("বড়")
#
# Backend হবে:
#
#     if x > 5:
#         print("বড়")
#
# ============================================================


class PythonCodeGenerator:

    # ========================================================
    # INITIALIZATION
    # ========================================================

    def __init__(self):

        self.lines = []

        self.indent_level = 0

        self.stack_names = set()

        self.queue_names = set()

        self.uses_deque = False


    # ========================================================
    # EMIT ONE LINE
    # ========================================================

    def emit(self, code=""):

        if code == "":

            self.lines.append("")

        else:

            indentation = "    " * self.indent_level

            self.lines.append(
                indentation + code
            )


    # ========================================================
    # SAFE ATTRIBUTE HELPER
    # ========================================================

    def get_attr(self, node, *names, default=None):

        for name in names:

            if hasattr(node, name):

                return getattr(
                    node,
                    name
                )

        return default


    # ========================================================
    # GENERATE COMPLETE PROGRAM
    # ========================================================

    def generate(self, node):

        # Reset state

        self.lines = []

        self.indent_level = 0

        self.stack_names = set()

        self.queue_names = set()

        self.uses_deque = False


        # Generate complete AST

        self.generate_node(node)


        # Queue ব্যবহার হলে import যোগ হবে

        if self.uses_deque:

            self.lines.insert(
                0,
                "from collections import deque"
            )

            self.lines.insert(
                1,
                ""
            )


        return "\n".join(
            self.lines
        )


    # ========================================================
    # GENERATE AST NODE
    # ========================================================

    def generate_node(self, node):

        if node is None:

            return


        # ====================================================
        # PROGRAM
        #
        # isinstance() mismatch হলেও
        # .statements structure দেখে Program handle করা হচ্ছে
        # ====================================================

        if isinstance(node, Program) or (
            type(node).__name__ == "Program"
            and hasattr(node, "statements")
        ):

            statements = self.get_attr(
                node,
                "statements",
                "body",
                default=[]
            )


            for statement in statements:

                self.generate_node(
                    statement
                )


            return


        # ====================================================
        # BLOCK
        # ====================================================

        if isinstance(node, Block) or (
            type(node).__name__ == "Block"
            and hasattr(node, "statements")
        ):

            statements = self.get_attr(
                node,
                "statements",
                "body",
                default=[]
            )


            if not statements:

                self.emit(
                    "pass"
                )

            else:

                for statement in statements:

                    self.generate_node(
                        statement
                    )


            return


        # ====================================================
        # VARIABLE DECLARATION
        #
        # পূর্ণসংখ্যা x = 10
        #
        # →
        #
        # x = 10
        # ====================================================

        if isinstance(node, VarDecl) or (
            type(node).__name__ == "VarDecl"
        ):

            name = self.get_attr(
                node,
                "name"
            )

            value_node = self.get_attr(
                node,
                "value"
            )

            value = self.generate_expression(
                value_node
            )


            self.emit(
                f"{name} = {value}"
            )


            return


        # ====================================================
        # ASSIGNMENT
        #
        # x = x + 5
        #
        # ছাত্র.নাম = "রহিম"
        # ====================================================

        if isinstance(node, Assignment) or (
            type(node).__name__ == "Assignment"
        ):

            target_node = self.get_attr(
                node,
                "target",
                "name"
            )

            value_node = self.get_attr(
                node,
                "value"
            )


            target = self.generate_expression(
                target_node
            )

            value = self.generate_expression(
                value_node
            )


            self.emit(
                f"{target} = {value}"
            )


            return


        # ====================================================
        # PRINT
        #
        # লেখো(x)
        #
        # →
        #
        # print(x)
        # ====================================================

        if isinstance(node, PrintStatement) or (
            type(node).__name__ == "PrintStatement"
        ):

            expression = self.get_attr(
                node,
                "expression",
                "value"
            )


            expression_code = (
                self.generate_expression(
                    expression
                )
            )


            self.emit(
                f"print({expression_code})"
            )


            return


        # ====================================================
        # IF
        #
        # যদি x > 5:
        #     ...
        #
        # →
        #
        # if x > 5:
        #     ...
        # ====================================================

        if isinstance(node, IfStatement) or (
            type(node).__name__ == "IfStatement"
        ):

            condition = self.get_attr(
                node,
                "condition"
            )

            then_branch = self.get_attr(
                node,
                "then_branch",
                "then_body",
                "body"
            )

            else_branch = self.get_attr(
                node,
                "else_branch",
                "else_body"
            )


            condition_code = (
                self.generate_expression(
                    condition
                )
            )


            self.emit(
                f"if {condition_code}:"
            )


            self.indent_level += 1

            self.generate_node(
                then_branch
            )

            self.indent_level -= 1


            if else_branch is not None:

                self.emit(
                    "else:"
                )

                self.indent_level += 1

                self.generate_node(
                    else_branch
                )

                self.indent_level -= 1


            return


        # ====================================================
        # WHILE
        #
        # যতক্ষণ x < 10:
        #     ...
        #
        # →
        #
        # while x < 10:
        #     ...
        # ====================================================

        if isinstance(node, WhileStatement) or (
            type(node).__name__ == "WhileStatement"
        ):

            condition = self.get_attr(
                node,
                "condition"
            )

            body = self.get_attr(
                node,
                "body"
            )


            condition_code = (
                self.generate_expression(
                    condition
                )
            )


            self.emit(
                f"while {condition_code}:"
            )


            self.indent_level += 1

            self.generate_node(
                body
            )

            self.indent_level -= 1


            return


        # ====================================================
        # FUNCTION DECLARATION
        #
        # ফাংশন যোগ(পূর্ণসংখ্যা a, পূর্ণসংখ্যা b):
        #     ফেরত a + b
        #
        # →
        #
        # def যোগ(a, b):
        #     return a + b
        # ====================================================

        if isinstance(node, FunctionDecl) or (
            type(node).__name__ == "FunctionDecl"
        ):

            name = self.get_attr(
                node,
                "name"
            )

            parameters = self.get_attr(
                node,
                "parameters",
                default=[]
            )

            body = self.get_attr(
                node,
                "body"
            )


            parameter_names = []


            for parameter in parameters:

                # Parameter tuple:
                # ("TYPE_INT", "a")

                if isinstance(
                    parameter,
                    tuple
                ):

                    if len(parameter) >= 2:

                        parameter_names.append(
                            parameter[1]
                        )


                elif isinstance(
                    parameter,
                    list
                ):

                    if len(parameter) >= 2:

                        parameter_names.append(
                            parameter[1]
                        )


                elif hasattr(
                    parameter,
                    "name"
                ):

                    parameter_names.append(
                        parameter.name
                    )


                else:

                    parameter_names.append(
                        str(parameter)
                    )


            parameter_text = ", ".join(
                parameter_names
            )


            self.emit(
                f"def {name}({parameter_text}):"
            )


            self.indent_level += 1


            if body is None:

                self.emit(
                    "pass"
                )

            else:

                self.generate_node(
                    body
                )


            self.indent_level -= 1


            return


        # ====================================================
        # RETURN
        #
        # ফেরত a + b
        #
        # →
        #
        # return a + b
        # ====================================================

        if isinstance(node, ReturnStatement) or (
            type(node).__name__ == "ReturnStatement"
        ):

            value_node = self.get_attr(
                node,
                "value"
            )


            if value_node is None:

                self.emit(
                    "return"
                )

            else:

                value = self.generate_expression(
                    value_node
                )

                self.emit(
                    f"return {value}"
                )


            return


        # ====================================================
        # STACK DECLARATION
        #
        # স্ট্যাক s
        #
        # →
        #
        # s = []
        # ====================================================

        if isinstance(node, StackDecl) or (
            type(node).__name__ == "StackDecl"
        ):

            name = self.get_attr(
                node,
                "name"
            )


            self.stack_names.add(
                name
            )


            self.emit(
                f"{name} = []"
            )


            return


        # ====================================================
        # QUEUE DECLARATION
        #
        # কিউ q
        #
        # →
        #
        # q = deque()
        # ====================================================

        if isinstance(node, QueueDecl) or (
            type(node).__name__ == "QueueDecl"
        ):

            name = self.get_attr(
                node,
                "name"
            )


            self.queue_names.add(
                name
            )

            self.uses_deque = True


            self.emit(
                f"{name} = deque()"
            )


            return


        # ====================================================
        # CLASS DECLARATION
        #
        # ক্লাস শিক্ষার্থী:
        #     ...
        #
        # →
        #
        # class শিক্ষার্থী:
        #     ...
        # ====================================================

        if isinstance(node, ClassDecl) or (
            type(node).__name__ == "ClassDecl"
        ):

            name = self.get_attr(
                node,
                "name"
            )

            properties = self.get_attr(
                node,
                "properties",
                "body",
                default=[]
            )


            self.emit(
                f"class {name}:"
            )


            self.indent_level += 1


            if not properties:

                self.emit(
                    "pass"
                )

            else:

                for property_node in properties:

                    self.generate_node(
                        property_node
                    )


            self.indent_level -= 1


            return


        # ====================================================
        # EXPRESSION STATEMENT
        #
        # s.ঠেলো(100)
        #
        # q.ঢোকাও("ক")
        # ====================================================

        if isinstance(node, ExpressionStatement) or (
            type(node).__name__ == "ExpressionStatement"
        ):

            expression = self.get_attr(
                node,
                "expression"
            )


            expression_code = (
                self.generate_expression(
                    expression
                )
            )


            self.emit(
                expression_code
            )


            return


        # ====================================================
        # FALLBACK
        #
        # কোনো node-এর statements থাকলে
        # সেটিকে block/program হিসেবে handle করবে
        # ====================================================

        if hasattr(node, "statements"):

            statements = node.statements


            for statement in statements:

                self.generate_node(
                    statement
                )


            return


        # ====================================================
        # UNKNOWN NODE
        # ====================================================

        raise TypeError(

            "Python Generator Error: Unsupported AST node "
            f"'{type(node).__name__}'"

        )


    # ========================================================
    # GENERATE EXPRESSION
    # ========================================================

    def generate_expression(self, node):

        if node is None:

            return "None"


        # ====================================================
        # NUMBER
        # ====================================================

        if isinstance(node, Number) or (
            type(node).__name__ == "Number"
        ):

            return str(
                self.get_attr(
                    node,
                    "value"
                )
            )


        # ====================================================
        # STRING
        # ====================================================

        if isinstance(node, String) or (
            type(node).__name__ == "String"
        ):

            return repr(
                self.get_attr(
                    node,
                    "value"
                )
            )


        # ====================================================
        # VARIABLE
        # ====================================================

        if isinstance(node, Variable) or (
            type(node).__name__ == "Variable"
        ):

            return str(
                self.get_attr(
                    node,
                    "name"
                )
            )


        # ====================================================
        # BINARY OPERATION
        # ====================================================

        if isinstance(node, BinaryOp) or (
            type(node).__name__ == "BinaryOp"
        ):

            left = self.generate_expression(
                self.get_attr(
                    node,
                    "left"
                )
            )

            operator = self.get_attr(
                node,
                "operator",
                "op"
            )

            right = self.generate_expression(
                self.get_attr(
                    node,
                    "right"
                )
            )


            return (
                f"({left} {operator} {right})"
            )


        # ====================================================
        # UNARY OPERATION
        # ====================================================

        if isinstance(node, UnaryOp) or (
            type(node).__name__ == "UnaryOp"
        ):

            operator = self.get_attr(
                node,
                "operator",
                "op"
            )

            operand = self.generate_expression(
                self.get_attr(
                    node,
                    "operand"
                )
            )


            return (
                f"({operator}{operand})"
            )


        # ====================================================
        # MEMBER ACCESS
        #
        # ছাত্র.নাম
        # ====================================================

        if isinstance(node, MemberAccess) or (
            type(node).__name__ == "MemberAccess"
        ):

            object_expr = self.get_attr(
                node,
                "object_expr",
                "object"
            )

            member = self.get_attr(
                node,
                "member",
                "name"
            )


            object_code = (
                self.generate_expression(
                    object_expr
                )
            )


            return (
                f"{object_code}.{member}"
            )


        # ====================================================
        # FUNCTION CALL
        # ====================================================

        if isinstance(node, FunctionCall) or (
            type(node).__name__ == "FunctionCall"
        ):

            return self.generate_function_call(
                node
            )


        # ====================================================
        # NEW OBJECT
        #
        # নতুন শিক্ষার্থী()
        #
        # →
        #
        # শিক্ষার্থী()
        # ====================================================

        if isinstance(node, NewObject) or (
            type(node).__name__ == "NewObject"
        ):

            class_name = self.get_attr(
                node,
                "class_name",
                "name"
            )


            return (
                f"{class_name}()"
            )


        # ====================================================
        # UNKNOWN EXPRESSION
        # ====================================================

        raise TypeError(

            "Python Generator Error: Unsupported expression "
            f"'{type(node).__name__}'"

        )


    # ========================================================
    # FUNCTION / METHOD CALL
    # ========================================================

    def generate_function_call(self, node):

        arguments = self.get_attr(
            node,
            "arguments",
            "args",
            default=[]
        )


        argument_codes = []


        for argument in arguments:

            argument_codes.append(

                self.generate_expression(
                    argument
                )

            )


        arguments_text = ", ".join(
            argument_codes
        )


        callee = self.get_attr(
            node,
            "name",
            "callee",
            "function"
        )


        # ====================================================
        # MEMBER FUNCTION CALL
        #
        # s.ঠেলো(100)
        #
        # q.ঢোকাও("ক")
        # ====================================================

        if isinstance(callee, MemberAccess) or (
            callee is not None
            and type(callee).__name__ == "MemberAccess"
        ):

            receiver = self.get_attr(
                callee,
                "object_expr",
                "object"
            )

            method_name = self.get_attr(
                callee,
                "member",
                "name"
            )


            receiver_code = (
                self.generate_expression(
                    receiver
                )
            )


            # =================================================
            # STACK
            # =================================================

            if (
                receiver_code in self.stack_names
            ):


                if method_name == "ঠেলো":

                    return (
                        f"{receiver_code}.append("
                        f"{arguments_text})"
                    )


                if method_name == "বের_করো":

                    return (
                        f"{receiver_code}.pop()"
                    )


                if method_name == "উপর_দেখো":

                    return (
                        f"{receiver_code}[-1]"
                    )


            # =================================================
            # QUEUE
            # =================================================

            if (
                receiver_code in self.queue_names
            ):


                if method_name == "ঢোকাও":

                    return (
                        f"{receiver_code}.append("
                        f"{arguments_text})"
                    )


                if method_name == "বের_করো":

                    return (
                        f"{receiver_code}.popleft()"
                    )


                if method_name == "সামনে":

                    return (
                        f"{receiver_code}[0]"
                    )


            # Normal member call

            return (
                f"{receiver_code}."
                f"{method_name}"
                f"({arguments_text})"
            )


        # ====================================================
        # NORMAL FUNCTION CALL
        # ====================================================

        if isinstance(callee, str):

            return (
                f"{callee}"
                f"({arguments_text})"
            )


        # ====================================================
        # CALLEE AS AST NODE
        # ====================================================

        if callee is not None:

            callee_code = (
                self.generate_expression(
                    callee
                )
            )


            return (
                f"{callee_code}"
                f"({arguments_text})"
            )


        raise TypeError(

            "Python Generator Error: "
            "Function call-এর callee পাওয়া যায়নি।"

        )


# ============================================================
# CELL 18 TEST
# ============================================================

print("=" * 60)

print(
    "CELL 18 — PYTHON CODE GENERATOR"
)

print("=" * 60)


# ============================================================
# FIND AST
# ============================================================

if "program_ast" in globals():

    ast_for_generation = program_ast


elif "ast_root" in globals():

    ast_for_generation = ast_root


elif "ast" in globals():

    ast_for_generation = ast


elif "program" in globals():

    ast_for_generation = program


else:

    raise RuntimeError(

        "CELL 18 Error: AST পাওয়া যায়নি। "
        "আগে Cell 14 run করুন।"

    )


# ============================================================
# GENERATE PYTHON CODE
# ============================================================

python_generator = PythonCodeGenerator()


generated_python_code = (
    python_generator.generate(
        ast_for_generation
    )
)


# ============================================================
# VALIDATION
# ============================================================

if not isinstance(
    generated_python_code,
    str
):

    raise TypeError(

        "CELL 18 Error: "
        "Generated Python code string নয়।"

    )


if not generated_python_code.strip():

    raise ValueError(

        "CELL 18 Error: "
        "Generated Python code খালি।"

    )


# ============================================================
# DISPLAY
# ============================================================

print()

print(
    "তৈরি হওয়া Python Backend Code:"
)

print("=" * 60)

print()

print(
    generated_python_code
)

print()

print("=" * 60)

print(
    "CELL 18 — PYTHON CODE GENERATION FINISHED! 🎉"
)

print("=" * 60)

CELL 18 — PYTHON CODE GENERATOR

তৈরি হওয়া Python Backend Code:

from collections import deque

x = 10
y = 2.5
নাম = 'বাংলাভাষা'
x = (x + (5 * 2))
print('বাংলা কম্পাইলারে স্বাগতম')
print(নাম)
print(x)
if (x > 5):
    print('মান বড়')
else:
    print('মান ছোট')
while (x < 25):
    print(x)
    x = (x + 1)
def যোগ(a, b):
    return (a + b)
ফল = যোগ(10, 20)
print(ফল)
s = []
s.append(100)
s.append(200)
print(s[-1])
print(s.pop())
q = deque()
q.append('ক')
q.append('খ')
print(q[0])
print(q.popleft())
class শিক্ষার্থী:
    নাম = 'অজানা'
    বয়স = 0
ছাত্র = শিক্ষার্থী()
ছাত্র.নাম = 'রহিম'
print(ছাত্র.নাম)

CELL 18 — PYTHON CODE GENERATION FINISHED! 🎉


In [19]:
# ============================================================
# CELL 19 — RUNTIME SAFETY + EXECUTION
# Bangla Python-Style Compiler
#
# Generated Python Code
#        ↓
# Safe Execution
#
# User Source Code : Bangla
# Program Output   : Bangla supported
# ============================================================


import io

from contextlib import (
    redirect_stdout,
    redirect_stderr
)


print("=" * 60)

print(
    "CELL 19 — রানটাইম সেফটি ও প্রোগ্রাম এক্সিকিউশন"
)

print("=" * 60)


# ============================================================
# CHECK CELL 18 OUTPUT
# ============================================================

if "generated_python_code" not in globals():

    raise RuntimeError(

        "CELL 19 Error: generated_python_code পাওয়া যায়নি। "
        "আগে Cell 18 run করুন।"

    )


if not isinstance(
    generated_python_code,
    str
):

    raise TypeError(

        "CELL 19 Error: "
        "generated_python_code string নয়।"

    )


if not generated_python_code.strip():

    raise ValueError(

        "CELL 19 Error: "
        "তৈরি হওয়া Python code খালি।"

    )


# ============================================================
# DISPLAY GENERATED BACKEND CODE
# ============================================================

print()

print(
    "তৈরি হওয়া Python Backend Code:"
)

print("-" * 60)

print(
    generated_python_code
)

print("-" * 60)


# ============================================================
# STEP 1 — SYNTAX CHECK
# ============================================================

print()

print(
    "ধাপ ১ — তৈরি হওয়া Python কোডের Syntax পরীক্ষা"
)


compiled_code = None

syntax_success = False

runtime_success = False


try:

    compiled_code = compile(

        generated_python_code,

        "<BanglaCompiler>",

        "exec"

    )


    syntax_success = True


    print(
        "✓ Syntax পরীক্ষা সফল"
    )


except SyntaxError as error:

    print(
        "✗ Syntax ত্রুটি পাওয়া গেছে"
    )

    print(
        f"লাইন নম্বর: {error.lineno}"
    )

    print(
        f"ত্রুটির বার্তা: {error.msg}"
    )


    if error.text:

        print(
            "সমস্যার Code: "
            f"{error.text.strip()}"
        )


# ============================================================
# STEP 2 — EXECUTION
# ============================================================

program_output = ""

program_error_output = ""

runtime_error = None


if compiled_code is not None:


    print()

    print(
        "ধাপ ২ — প্রোগ্রাম চালানো হচ্ছে"
    )

    print("-" * 60)


    # ========================================================
    # RUNTIME ENVIRONMENT
    # ========================================================

    runtime_environment = {

        "__name__": "__main__"

    }


    # ========================================================
    # OUTPUT CAPTURE
    # ========================================================

    captured_stdout = io.StringIO()

    captured_stderr = io.StringIO()


    try:

        with redirect_stdout(
            captured_stdout
        ), redirect_stderr(
            captured_stderr
        ):

            exec(

                compiled_code,

                runtime_environment,

                runtime_environment

            )


        runtime_success = True


    # ========================================================
    # DIVISION BY ZERO
    # ========================================================

    except ZeroDivisionError:

        runtime_error = (

            "শূন্য দিয়ে ভাগ করা যাবে না।"

        )


    # ========================================================
    # EMPTY STACK / QUEUE
    #
    # Python list.pop()
    # deque.popleft()
    # q[0]
    # s[-1]
    # ========================================================

    except IndexError as error:

        runtime_error = (

            "খালি Stack অথবা Queue থেকে "
            "মান নেওয়ার চেষ্টা করা হয়েছে। "
            f"বিস্তারিত: {error}"

        )


    # ========================================================
    # TYPE ERROR
    # ========================================================

    except TypeError as error:

        runtime_error = (

            f"টাইপ ত্রুটি: {error}"

        )


    # ========================================================
    # UNDEFINED NAME
    # ========================================================

    except NameError as error:

        runtime_error = (

            f"অঘোষিত নাম ব্যবহার করা হয়েছে: {error}"

        )


    # ========================================================
    # PROPERTY ERROR
    # ========================================================

    except AttributeError as error:

        runtime_error = (

            "অবৈধ Property অথবা Member access: "
            f"{error}"

        )


    # ========================================================
    # VALUE ERROR
    # ========================================================

    except ValueError as error:

        runtime_error = (

            f"অবৈধ মান: {error}"

        )


    # ========================================================
    # GENERAL ERROR
    # ========================================================

    except Exception as error:

        runtime_error = (

            f"{type(error).__name__}: "
            f"{error}"

        )


    finally:

        program_output = (
            captured_stdout.getvalue()
        )


        program_error_output = (
            captured_stderr.getvalue()
        )


# ============================================================
# DISPLAY PROGRAM OUTPUT
# ============================================================

if compiled_code is not None:


    print()

    print(
        "প্রোগ্রামের আউটপুট:"
    )

    print("=" * 60)


    if program_output.strip():

        print(
            program_output,
            end=""
        )


    else:

        print(
            "(কোনো আউটপুট নেই)"
        )


    print("=" * 60)


# ============================================================
# DISPLAY RUNTIME ERROR
# ============================================================

if runtime_error is not None:


    print()

    print(
        "রানটাইম ত্রুটি:"
    )

    print("-" * 60)

    print(
        runtime_error
    )

    print("-" * 60)


# ============================================================
# DISPLAY STDERR
# ============================================================

if program_error_output.strip():

    print()

    print(
        "Python Runtime Error Output:"
    )

    print("-" * 60)

    print(
        program_error_output
    )

    print("-" * 60)


# ============================================================
# SUMMARY
# ============================================================

print()

print(
    "রানটাইম সেফটি সারাংশ"
)

print("-" * 60)


if syntax_success:

    print(
        "✓ Python syntax পরীক্ষা"
    )

else:

    print(
        "✗ Python syntax পরীক্ষা ব্যর্থ"
    )


print(
    "✓ শূন্য দিয়ে ভাগের ত্রুটি ধরা"
)

print(
    "✓ খালি Stack handling"
)

print(
    "✓ খালি Queue handling"
)

print(
    "✓ Type error handling"
)

print(
    "✓ Undefined name handling"
)

print(
    "✓ Property access error handling"
)

print(
    "✓ সাধারণ runtime error handling"
)

print("-" * 60)


# ============================================================
# FINAL RESULT
# ============================================================

if runtime_success:

    print()

    print(
        "চূড়ান্ত ফলাফল: "
        "প্রোগ্রাম সফলভাবে চালানো হয়েছে! ✅"
    )


elif compiled_code is None:

    print()

    print(
        "চূড়ান্ত ফলাফল: "
        "Syntax ত্রুটির কারণে প্রোগ্রাম চালানো হয়নি। ⚠️"
    )


else:

    print()

    print(
        "চূড়ান্ত ফলাফল: "
        "প্রোগ্রাম নিরাপদভাবে বন্ধ হয়েছে। ⚠️"
    )


print()

print("=" * 60)

print(
    "CELL 19 — রানটাইম সেফটি ও এক্সিকিউশন শেষ! 🎉"
)

print("=" * 60)

CELL 19 — রানটাইম সেফটি ও প্রোগ্রাম এক্সিকিউশন

তৈরি হওয়া Python Backend Code:
------------------------------------------------------------
from collections import deque

x = 10
y = 2.5
নাম = 'বাংলাভাষা'
x = (x + (5 * 2))
print('বাংলা কম্পাইলারে স্বাগতম')
print(নাম)
print(x)
if (x > 5):
    print('মান বড়')
else:
    print('মান ছোট')
while (x < 25):
    print(x)
    x = (x + 1)
def যোগ(a, b):
    return (a + b)
ফল = যোগ(10, 20)
print(ফল)
s = []
s.append(100)
s.append(200)
print(s[-1])
print(s.pop())
q = deque()
q.append('ক')
q.append('খ')
print(q[0])
print(q.popleft())
class শিক্ষার্থী:
    নাম = 'অজানা'
    বয়স = 0
ছাত্র = শিক্ষার্থী()
ছাত্র.নাম = 'রহিম'
print(ছাত্র.নাম)
------------------------------------------------------------

ধাপ ১ — তৈরি হওয়া Python কোডের Syntax পরীক্ষা
✓ Syntax পরীক্ষা সফল

ধাপ ২ — প্রোগ্রাম চালানো হচ্ছে
------------------------------------------------------------

প্রোগ্রামের আউটপুট:
বাংলা কম্পাইলারে স্বাগতম
বাংলাভাষা
20
মান বড়
20
21
22
23
24
30
200
200
ক
ক
রহ

In [20]:
# ============================================================
# CELL 20 — BANGLA USER / DEMO SOURCE
# ============================================================
#
# এখানে শুধু USER_SOURCE-এর ভিতরে নিজের
# Bangla Python-style program লিখবে।
#
# উদাহরণ:
#
# পূর্ণসংখ্যা x = 10
# দশমিক y = 2.5
# স্ট্রিং নাম = "রহিম"
#
# x = x + 5
#
# লেখো(x)
#
# যদি x > 5:
#     লেখো("বড়")
# নাহলে:
#     লেখো("ছোট")
#
# যতক্ষণ x < 10:
#     লেখো(x)
#     x = x + 1
#
# ============================================================


USER_SOURCE = '''

ক্লাস শিক্ষার্থী:
    স্ট্রিং নাম = "অজানা"
    পূর্ণসংখ্যা বয়স = 0

শিক্ষার্থী ছাত্র = নতুন শিক্ষার্থী()

ছাত্র.নাম = "রহিম"
ছাত্র.বয়স = 20

লেখো(ছাত্র.নাম)
লেখো(ছাত্র.বয়স)

'''

In [21]:
# ============================================================
# CELL 21 — FINAL BANGLA COMPILER PIPELINE
# ============================================================
#
# Pipeline:
#
# Bangla Source
#       ↓
# Source Review
#       ↓
# Lexer
#       ↓
# Tokens
#       ↓
# Parser
#       ↓
# AST
#       ↓
# Semantic Analysis
#       ↓
# TAC
#       ↓
# Python Code Generation
#       ↓
# Python Syntax Check
#       ↓
# Execution
#
# ============================================================


# ============================================================
# HEADER
# ============================================================

print()
print("=" * 70)
print("       বাংলা পাইথন-স্টাইল কম্পাইলার")
print("=" * 70)


# ============================================================
# STEP 1 — SOURCE REVIEW
# ============================================================

print()
print("১. তোমার লেখা বাংলা সোর্স কোড")
print("-" * 70)

print(USER_SOURCE)


# ============================================================
# STEP 2 — LEXER
# ============================================================

print()
print("=" * 70)
print("২. লেক্সার — টোকেন তৈরি")
print("-" * 70)


try:

    compiler_lexer = Lexer(USER_SOURCE)

    compiler_tokens = compiler_lexer.tokenize()


    print()

    print(
        f"মোট টোকেন: {len(compiler_tokens)}"
    )


    for token in compiler_tokens:

        print(token)


except Exception as error:

    print()

    print("❌ লেক্সার ত্রুটি!")

    print(
        type(error).__name__
        + ": "
        + str(error)
    )

    raise


# ============================================================
# STEP 3 — PARSER
# ============================================================

print()
print("=" * 70)
print("৩. পার্সার — AST তৈরি")
print("-" * 70)


try:

    compiler_parser = Parser(
        compiler_tokens
    )


    compiler_ast = (
        compiler_parser.parse()
    )


    print()

    print("তৈরি হওয়া AST:")

    print()

    print(
        compiler_ast
    )


    # Parser errors check

    if hasattr(
        compiler_parser,
        "errors"
    ):

        if compiler_parser.errors:

            print()

            print(
                "❌ Parser Errors:"
            )


            for error in compiler_parser.errors:

                print(error)


        else:

            print()

            print(
                "✅ Parser Error পাওয়া যায়নি।"
            )


    else:

        print()

        print(
            "✅ Parser সম্পন্ন হয়েছে।"
        )


except Exception as error:

    print()

    print("❌ পার্সার ত্রুটি!")

    print(
        type(error).__name__
        + ": "
        + str(error)
    )

    raise


# ============================================================
# STEP 4 — SEMANTIC ANALYSIS
# ============================================================

print()
print("=" * 70)
print("৪. সেমান্টিক বিশ্লেষণ")
print("-" * 70)


try:

    compiler_symbol_table = (
        SymbolTable()
    )


    compiler_semantic = (
        SemanticAnalyzer(
            compiler_symbol_table
        )
    )


    if hasattr(
        compiler_semantic,
        "analyze"
    ):

        compiler_semantic.analyze(
            compiler_ast
        )


    else:

        raise RuntimeError(
            "SemanticAnalyzer-এ analyze() method পাওয়া যায়নি।"
        )


    print()


    if (
        hasattr(
            compiler_semantic,
            "errors"
        )
        and compiler_semantic.errors
    ):


        print(
            "❌ Semantic Errors পাওয়া গেছে:"
        )

        print()


        for error in compiler_semantic.errors:

            print(error)


        raise RuntimeError(
            "Semantic Analysis ব্যর্থ হয়েছে।"
        )


    else:

        print(
            "✅ Semantic Analysis সফল!"
        )


except Exception as error:

    print()


    if (
        "compiler_semantic" in globals()
        and hasattr(
            compiler_semantic,
            "errors"
        )
        and compiler_semantic.errors
    ):

        print(
            "❌ প্রোগ্রামে semantic সমস্যা আছে।"
        )


    else:

        print(
            "❌ Semantic Analysis ত্রুটি!"
        )

        print(
            type(error).__name__
            + ": "
            + str(error)
        )


    raise


# ============================================================
# STEP 5 — TAC GENERATION
# ============================================================

print()
print("=" * 70)
print("৫. থ্রি অ্যাড্রেস কোড (TAC)")
print("-" * 70)


try:

    compiler_tac_generator = (
        TACGenerator()
    )


    compiler_tac_generator.generate(
        compiler_ast
    )


    compiler_tac_code = (
        compiler_tac_generator.get_code()
    )


    print()


    if compiler_tac_code:


        for instruction in compiler_tac_code:

            print(
                instruction
            )


    else:

        print(
            "এই প্রোগ্রামের জন্য কোনো TAC instruction তৈরি হয়নি।"
        )


except Exception as error:

    print()

    print(
        "❌ TAC Generation ত্রুটি!"
    )

    print(
        type(error).__name__
        + ": "
        + str(error)
    )

    raise


# ============================================================
# STEP 6 — PYTHON CODE GENERATION
# ============================================================

print()
print("=" * 70)
print("৬. তৈরি হওয়া Python Backend Code")
print("-" * 70)


try:

    compiler_generator = (
        PythonCodeGenerator()
    )


    compiler_python_code = (
        compiler_generator.generate(
            compiler_ast
        )
    )


    if not isinstance(
        compiler_python_code,
        str
    ):

        raise TypeError(
            "Python generator string code তৈরি করেনি।"
        )


    if not compiler_python_code.strip():

        raise ValueError(
            "Generated Python code খালি।"
        )


    print()

    print(
        compiler_python_code
    )


except Exception as error:

    print()

    print(
        "❌ Python Code Generation ত্রুটি!"
    )

    print(
        type(error).__name__
        + ": "
        + str(error)
    )

    raise


# ============================================================
# STEP 7 — PYTHON SYNTAX CHECK
# ============================================================

print()
print("=" * 70)
print("৭. তৈরি হওয়া Python Code যাচাই")
print("-" * 70)


try:

    compile(
        compiler_python_code,
        "<BanglaCompiler>",
        "exec"
    )


    print()

    print(
        "✅ Generated Python code syntax সঠিক।"
    )


except SyntaxError as error:

    print()

    print(
        "❌ Generated Python Syntax Error!"
    )

    print(
        str(error)
    )

    raise


# ============================================================
# STEP 8 — FINAL EXECUTION
# ============================================================

print()
print("=" * 70)
print("৮. চূড়ান্ত প্রোগ্রাম আউটপুট")
print("-" * 70)

print()


try:

    compiler_runtime = {

        "__name__": "__main__"

    }


    exec(
        compiler_python_code,
        compiler_runtime,
        compiler_runtime
    )


except ZeroDivisionError:

    print()

    print(
        "❌ রানটাইম ত্রুটি: শূন্য দিয়ে ভাগ করা যায় না।"
    )


except IndexError:

    print()

    print(
        "❌ রানটাইম ত্রুটি: Stack অথবা Queue খালি।"
    )


except TypeError as error:

    print()

    print(
        "❌ রানটাইম Type Error:"
    )

    print(
        str(error)
    )


except Exception as error:

    print()

    print(
        "❌ রানটাইম ত্রুটি:"
    )

    print(
        type(error).__name__
        + ": "
        + str(error)
    )


# ============================================================
# FINAL MESSAGE
# ============================================================

print()

print("=" * 70)

print(
    "বাংলা কম্পাইলার রান সম্পন্ন! 🎉"
)

print("=" * 70)


       বাংলা পাইথন-স্টাইল কম্পাইলার

১. তোমার লেখা বাংলা সোর্স কোড
----------------------------------------------------------------------


ক্লাস শিক্ষার্থী:
    স্ট্রিং নাম = "অজানা"
    পূর্ণসংখ্যা বয়স = 0

শিক্ষার্থী ছাত্র = নতুন শিক্ষার্থী()

ছাত্র.নাম = "রহিম"
ছাত্র.বয়স = 20

লেখো(ছাত্র.নাম)
লেখো(ছাত্র.বয়স)



২. লেক্সার — টোকেন তৈরি
----------------------------------------------------------------------

মোট টোকেন: 51
Token(CLASS, 'ক্লাস', line=3, column=1)
Token(IDENTIFIER, 'শিক্ষার্থী', line=3, column=7)
Token(COLON, ':', line=3, column=17)
Token(NEWLINE, '', line=3, column=18)
Token(INDENT, '', line=4, column=1)
Token(TYPE_STRING, 'স্ট্রিং', line=4, column=5)
Token(IDENTIFIER, 'নাম', line=4, column=13)
Token(ASSIGN, '=', line=4, column=17)
Token(STRING, 'অজানা', line=4, column=19)
Token(NEWLINE, '', line=4, column=26)
Token(TYPE_INT, 'পূর্ণসংখ্যা', line=5, column=5)
Token(IDENTIFIER, 'বয়স', line=5, column=17)
Token(ASSIGN, '=', line=5, column=21)
Token(INTEGER, '0', line=5, co